# PropTech 360 | Data Engineering Graduation Capstone
### Industry: Commercial real estate and property management · Level: Intermediate · Total: 100 points

**Student name:** ____________________  **Cohort:** _______APRIL________  **Submission date:** ______OCT 10, 2026_______

> **Assessment notebook, not a worked solution.** Execute the dataset generator, complete the `TODO` tasks independently, and provide evidence of local and AWS runs. Use fictional, synthetic records only.

**Scenario.** MetroSpace Property Group manages a portfolio of commercial buildings. Property, unit, lease, repair, and energy records arrive as disconnected CSV exports. Management cannot reliably compare occupancy, earned rental revenue, maintenance SLA performance, or energy use across buildings. You have been hired to design a reproducible analytics pipeline and explain its reliability and operating cost.

**Mission.** Ingest raw source files; detect and address quality defects; transform data at scale; model and query a PostgreSQL mart; then land curated outputs in an AWS serverless analytics architecture. Your solution must be rerunnable without silently doubling records.

## 1. Five assessed topics and end-to-end architecture

1. **Python ingestion and file handling** — load five raw CSV extracts and record ingestion metadata.
2. **Pandas ETL and data quality** — profile, validate, quarantine, and clean datasets with explicit quality rules.
3. **PySpark transformations** — derive monthly occupancy, rental revenue, maintenance SLA, and energy metrics.
4. **PostgreSQL, SQL, and analytical modeling** — implement a dimensional mart and solve business questions.
5. **AWS serverless pipeline** — S3 raw/curated zones, Lambda, Glue Data Catalog, Athena, IAM, and CloudWatch.

**Reference architecture**

`CSV source extracts → Python intake / raw zone → Pandas validation + quarantine → PySpark curated Parquet → PostgreSQL dimensional mart → SQL reporting`

`                                                                     ↘ S3 curated → Glue Catalog → Athena`  
`S3 raw → S3 event → Lambda (validation / orchestration metadata) → S3 audit + CloudWatch`

**Design constraint:** Do not run Spark inside Lambda. Lambda handles lightweight event validation or metadata; perform the Spark workload separately, locally or on an appropriately provisioned service. Do not claim that S3 automatically refreshes PostgreSQL.

## 2. Business requirements and KPI definitions

Report results **per property and month (January–June 2026)** and provide portfolio-wide totals where appropriate.

- **Occupancy rate:** occupied units / total active units at the **month end**. A unit is occupied when at least one valid lease has `start_date <= month_end <= end_date` (or `end_date` is missing). Flag conflicting simultaneously active leases for the same unit instead of counting the unit twice.
- **Earned monthly rent (simplified):** sum of contractual `monthly_rent_usd` for leases active on the **last day of that month**. This is an analytical proxy, **not cash collected, prorated rent, or recognized accounting revenue**.
- **Maintenance SLA compliance:** share of resolved work orders whose `response_hours` is within target: CRITICAL ≤ 4 h, HIGH ≤ 12 h, MEDIUM ≤ 48 h, LOW ≤ 72 h. Exclude unresolved orders from the compliance denominator and report their count separately. Define and document any additional exclusions.
- **Monthly energy intensity:** sum of valid `energy_kwh` / total property floor area in m²; state whether your chosen denominator includes all or occupied units. Never divide by zero.

**Stakeholders:** portfolio director, operations manager, finance analyst, and cloud platform team. **Grain discipline:** do not join work-order line items directly to lease line items when computing rent; this can multiply totals.

## 3. Environment and setup

**Local:** Python 3.12 recommended for compatibility; pandas, numpy, pyspark, pyarrow, sqlalchemy, psycopg (or psycopg2), Jupyter; PostgreSQL; Java matching your installed PySpark version. Use VS Code or Jupyter. Match PySpark and Java versions to their official compatibility guidance. A local PostgreSQL database and AWS account with appropriate permissions are required for their respective implementation tasks.

**AWS:** S3, Lambda, Glue Data Catalog (crawler or explicit table definitions), Athena, IAM, and CloudWatch. Use a dedicated learning account or permitted lab. AWS resources can incur charges; set a budget alert, avoid sensitive data, restrict IAM privileges, and delete resources after assessment. **Never embed access keys or passwords in notebook outputs or Git.**

**Suggested repository structure**
```text
proptech360-capstone/
  capstone.ipynb
  requirements.txt
  README.md
  data/raw/              # five generated source files
  data/quarantine/       # rejected records + reasons
  data/curated/          # partitioned Parquet + audit/summary
  src/                   # reusable ingestion, validation and ETL modules
  sql/                   # DDL, upserts, reporting queries
  aws/                   # Lambda handler + IAM policy description + deployment steps
  tests/                 # checks for quality rules and KPI logic
  evidence/              # redacted run screenshots / Athena results
```

**Installation (run in the notebook's Python environment):** `python -m pip install pandas numpy pyspark pyarrow sqlalchemy psycopg[binary] pytest boto3` (or pin compatible versions in `requirements.txt`). The next cell checks common local imports; install packages before running it.

In [1]:
from pathlib import Path

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebook":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
STAGING_DIR = DATA_DIR / "staging"
QUARANTINE_DIR = DATA_DIR / "quarantine"
CURATED_DIR = DATA_DIR / "curated"

print("Working directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)

Working directory: c:\DataEngineeringProjects\proptech360-project\notebook
Project root: c:\DataEngineeringProjects\proptech360-project
Raw directory: c:\DataEngineeringProjects\proptech360-project\data\raw


In [2]:
print("RAW_DIR exists:", RAW_DIR.exists())

files = list(RAW_DIR.glob("*.csv"))

print("CSV files found:", len(files))

for file in files:
    print(file.name)

RAW_DIR exists: True
CSV files found: 5
leases.csv
meter_readings.csv
properties.csv
units.csv
work_orders.csv


In [3]:
# Environment check: record the versions in your README.
import sys
import importlib.metadata as metadata
print('Python:', sys.version.split()[0])
for package in ['pandas', 'numpy', 'pyspark', 'pyarrow', 'sqlalchemy', 'psycopg', 'boto3']:
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: NOT INSTALLED (install before its assessed section)')

Python: 3.12.10
pandas: 2.2.3
numpy: 2.5.3
pyspark: 4.2.0
pyarrow: 25.0.1
sqlalchemy: 2.1.3
psycopg: 3.3.6
boto3: 1.43.108


## 4. Generate the synthetic source extracts

Run the generator **once or rerun with the same seed**; it overwrites the same five source files deterministically. Dates and properties are fictional. Deliberately injected quality issues include repeated work orders, negative meter readings, invalid foreign keys, missing response durations, and duplicate unit identifiers. **Do not edit raw files manually**; preserve them as immutable source inputs and fix issues in your pipeline.

**Source contract**

| File | Grain | Primary identifier | Important fields |
|---|---|---|---|
| `properties.csv` | one property | `property_id` | name, city |
| `units.csv` | one unit | `unit_id` | property_id, floor_area_sqm, active |
| `leases.csv` | one lease | `lease_id` | unit_id, start/end dates, monthly_rent_usd |
| `work_orders.csv` | one work order | `work_order_id` | unit_id, opened/closed dates, priority, response_hours, cost_usd |
| `meter_readings.csv` | unit-month | `reading_id` | unit_id, reading_month, energy_kwh |

The file formats intentionally contain dates as text; you must parse and validate them. The generator is **test fixture setup**, not a graded solution.

In [17]:
from pathlib import Path
import csv
import random
from datetime import date, timedelta

rng = random.Random(3602026)
root = Path('data')
raw = root / 'raw'
raw.mkdir(parents=True, exist_ok=True)
for folder in ['quarantine', 'curated']:
    (root / folder).mkdir(parents=True, exist_ok=True)

def write_csv(name, rows):
    path = raw / name
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    print(f'{path}: {len(rows):,} rows')

cities = ['Lagos', 'Abuja', 'Ibadan', 'Port Harcourt']
properties = [dict(property_id=f'P{i:03}', property_name=f'MetroSpace Tower {i:02}',
                   city=cities[(i-1) % len(cities)]) for i in range(1, 13)]
units = []
for p in properties:
    for j in range(1, 21):
        units.append(dict(unit_id=f"{p['property_id']}-U{j:03}", property_id=p['property_id'],
                          floor_area_sqm=rng.choice([45, 60, 75, 90, 120, 150]), active=1))
leases = []
for i, unit in enumerate(units, start=1):
    if rng.random() < 0.22:  # intentionally vacant unit
        continue
    start = date(2025, rng.randint(1, 12), rng.randint(1, 25))
    end = date(2026, rng.randint(3, 12), rng.randint(1, 28)) if rng.random() < 0.27 else None
    leases.append(dict(lease_id=f'L{i:05}', unit_id=unit['unit_id'],
                       start_date=start.isoformat(), end_date=end.isoformat() if end else '',
                       monthly_rent_usd=rng.choice([650, 800, 975, 1200, 1500, 1800, 2200])))
priorities = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
work_orders = []
for i in range(1, 1501):
    unit = rng.choice(units)
    opened = date(2026, rng.randint(1, 6), rng.randint(1, 28))
    resolved = rng.random() < 0.82
    response = round(rng.uniform(1, 96), 1)
    work_orders.append(dict(work_order_id=f'WO{i:06}', unit_id=unit['unit_id'],
        opened_date=opened.isoformat(), closed_date=(opened + timedelta(days=rng.randint(0, 15))).isoformat() if resolved else '',
        priority=rng.choice(priorities), response_hours=response if resolved else '',
        cost_usd=round(rng.uniform(20, 1250), 2), status='RESOLVED' if resolved else 'OPEN'))
months = [f'2026-{m:02}-01' for m in range(1, 7)]
readings = []
for unit in units:
    for month in months:
        readings.append(dict(reading_id=f'R{len(readings)+1:06}', unit_id=unit['unit_id'],
                             reading_month=month, energy_kwh=round(rng.uniform(100, 1250), 2)))
# Known intentionally dirty records; do not remove these from the raw source.
work_orders.append(work_orders[8].copy())                     # exact duplicate
work_orders[34]['unit_id'] = 'P999-U999'                       # unknown parent
work_orders[52]['response_hours'] = ''                         # resolved with missing response
work_orders[63]['cost_usd'] = '-300'                           # invalid negative repair cost
readings[17]['energy_kwh'] = '-45'                             # invalid negative consumption
readings[34]['unit_id'] = 'P999-U999'                           # unknown unit
units.append(units[4].copy())                                 # duplicate identifier
for name, rows in [('properties.csv', properties), ('units.csv', units), ('leases.csv', leases),
                   ('work_orders.csv', work_orders), ('meter_readings.csv', readings)]:
    write_csv(name, rows)
print('Raw fixture ready. Retain a copy before any transformations.')

data\raw\properties.csv: 12 rows
data\raw\units.csv: 241 rows
data\raw\leases.csv: 198 rows
data\raw\work_orders.csv: 1,501 rows
data\raw\meter_readings.csv: 1,440 rows
Raw fixture ready. Retain a copy before any transformations.


In [5]:
import os

print(os.getcwd())

c:\DataEngineeringProjects\proptech360-project\notebook


In [6]:
files = list(RAW_DIR.glob("*.csv"))
print("CSV files found:", len(files))

CSV files found: 5


## 5. Topic 1 — Python ingestion and reproducibility (15 points)

**Tasks:** (a) create a reusable loader for the five source files; (b) validate file existence, expected columns, and nonempty inputs; (c) record filename, row count, ingest timestamp UTC, and SHA-256 digest in an audit manifest; (d) make repeated runs safe—raw source files must remain unchanged; (e) expose clean error messages for a missing file or missing required column.

**Evidence:** reusable `src/ingest.py`, audit manifest, and a test showing controlled failure for a nonexistent path. Use `pathlib`, `hashlib`, `datetime`, `csv` and/or pandas. **Do not** use `print()` as your only error handling strategy.

In [7]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 2.2.3
NumPy: 2.5.3


In [8]:
import pandas as pd
import hashlib

from pathlib import Path
from datetime import datetime, timezone

In [ ]:
# STUDENT TASK 1: implement the loader; use the generated files under data/raw.
from pathlib import Path
RAW_DIR = Path('data/raw')
EXPECTED_COLUMNS = {
    'properties.csv': {'property_id', 'property_name', 'city'},
    'units.csv': {'unit_id', 'property_id', 'floor_area_sqm', 'active'},
    'leases.csv': {'lease_id', 'unit_id', 'start_date', 'end_date', 'monthly_rent_usd'},
    'work_orders.csv': {'work_order_id', 'unit_id', 'opened_date', 'closed_date', 'priority', 'response_hours', 'cost_usd', 'status'},
    'meter_readings.csv': {'reading_id', 'unit_id', 'reading_month', 'energy_kwh'},
}

def ingest_source(path: Path, expected_columns: set):
    """Return a DataFrame and manifest entry; raise a descriptive error on bad input."""
    # TODO: validate file and columns; load with stable dtypes; compute hash and UTC timestamp.
    raise NotImplementedError('Student implementation required')

# TODO: iterate through EXPECTED_COLUMNS and populate datasets + ingestion_manifest.
# TODO: save the manifest to data/curated/ingestion_manifest.json.

### Solving Topic 1

In [9]:
# STUDENT TASK 1: implement the loader; use the generated files under data/raw.
from pathlib import Path
RAW_DIR = RAW_DIR
EXPECTED_COLUMNS = {
    'properties.csv': {'property_id', 'property_name', 'city'},
    'units.csv': {'unit_id', 'property_id', 'floor_area_sqm', 'active'},
    'leases.csv': {'lease_id', 'unit_id', 'start_date', 'end_date', 'monthly_rent_usd'},
    'work_orders.csv': {'work_order_id', 'unit_id', 'opened_date', 'closed_date', 'priority', 'response_hours', 'cost_usd', 'status'},
    'meter_readings.csv': {'reading_id', 'unit_id', 'reading_month', 'energy_kwh'},
}

def ingest_source(path: Path, expected_columns: set):
    # Confirm that the source file exists
    if not path.exists():
        raise FileNotFoundError(f"Source file not found: {path}")

    # Read the CSV file
    df = pd.read_csv(path)

    # Reject an empty input
    if df.empty:
        raise ValueError(f"Source file is empty: {path.name}")

    # Check required columns
    missing_columns = expected_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{path.name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    # Generate SHA-256 fingerprint
    sha256 = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha256.update(chunk)

    # Record audit information
    metadata = {
        "filename": path.name,
        "row_count": len(df),
        "ingested_at_utc": datetime.now(timezone.utc).isoformat(),
        "sha256": sha256.hexdigest()
    }

    return df, metadata

#### Testing One File

In [10]:
properties_df, properties_metadata = ingest_source(
    RAW_DIR / "properties.csv",
    EXPECTED_COLUMNS["properties.csv"]
)

print(properties_df.head())
print()
print(properties_metadata)

  property_id        property_name           city
0        P001  MetroSpace Tower 01          Lagos
1        P002  MetroSpace Tower 02          Abuja
2        P003  MetroSpace Tower 03         Ibadan
3        P004  MetroSpace Tower 04  Port Harcourt
4        P005  MetroSpace Tower 05          Lagos

{'filename': 'properties.csv', 'row_count': 12, 'ingested_at_utc': '2026-10-15T09:50:28.840104+00:00', 'sha256': 'c9865bcf2b142026c5dd05547f9c14d3e11f5a27a3e22bca52f5efe0c795ef19'}


#### ingesting all five source files and build the audit manifest.

In [12]:
datasets = {}
ingestion_manifest = []

for filename, expected_columns in EXPECTED_COLUMNS.items():

    df, metadata = ingest_source(
        RAW_DIR / filename,
        expected_columns
    )

    datasets[filename] = df
    ingestion_manifest.append(metadata)

    print(
        f"Loaded {filename}: "
        f"{metadata['row_count']:,} rows"
    )

Loaded properties.csv: 12 rows
Loaded units.csv: 241 rows
Loaded leases.csv: 198 rows
Loaded work_orders.csv: 1,501 rows
Loaded meter_readings.csv: 1,440 rows


In [13]:
# Inspect the ingestion manifest
manifest_df = pd.DataFrame(ingestion_manifest)
manifest_df

,filename,row_count,ingested_at_utc,sha256
0,properties.csv,12,2026-10-15T09:50:42.333668+00:00,c9865bcf2b142026c5dd05547f9c14d3e11f5a27a3e22b...
1,units.csv,241,2026-10-15T09:50:42.338601+00:00,f9bcef3d08d4662c270bfff385299306995ed6ba3f0c54...
2,leases.csv,198,2026-10-15T09:50:42.345405+00:00,b6a8e10a11dfbca4358d5a9d60ad4709bc6511417137ae...
3,work_orders.csv,1501,2026-10-15T09:50:42.358154+00:00,835c05c0410f7a481d71f4efa8213520d012688ebb8b75...
4,meter_readings.csv,1440,2026-10-15T09:50:42.367902+00:00,93684a8ed99f3e4f368646d2f766a41805d54e7db5ef43...


In [14]:
# Save the manifest to data/curated/ingestion_manifest.json
import json

manifest_path = CURATED_DIR / "ingestion_manifest.json"

with manifest_path.open("w", encoding="utf-8") as file:
    json.dump(
        ingestion_manifest,
        file,
        indent=4
    )

print(f"Manifest saved to: {manifest_path}")

Manifest saved to: c:\DataEngineeringProjects\proptech360-project\data\curated\ingestion_manifest.json


In [15]:
try:
    ingest_source(
        RAW_DIR / "missing_file.csv",
        {"example_column"}
    )

except FileNotFoundError as error:
    print("Controlled failure:")
    print(error)

Controlled failure:
Source file not found: c:\DataEngineeringProjects\proptech360-project\data\raw\missing_file.csv


In [16]:
test_file = PROJECT_ROOT / "data" / "bad_test.csv"

pd.DataFrame({
    "property_id": ["P999"],
    "property_name": ["Test Property"]
}).to_csv(test_file, index=False)

In [17]:
try:
    ingest_source(
        test_file,
        EXPECTED_COLUMNS["properties.csv"]
    )

except ValueError as error:
    print("Controlled failure:")
    print(error)

Controlled failure:
bad_test.csv is missing required columns: ['city']


In [18]:
test_file.unlink()

Topic 1 is complete.
Your Topic 1 checklist should be:
- reusable loader works
- all five datasets load
- file existence validated
- empty input validated
- required columns validated
- row counts recorded
- UTC timestamps recorded
- SHA-256 digests generated
- audit manifest saved
- missing-file failure demonstrated
- missing-column failure demonstrated
- raw source files unchanged

## 6. Topic 2 — Pandas validation, cleaning, and quarantine (20 points)

**Tasks:** Profile null counts, duplicate identifiers, date parse failures, invalid category values, non-positive areas, negative money/energy, broken foreign keys, invalid chronological order, and overlapping leases. Implement a **documented policy** for each rule: fail the run, quarantine the row, correct deterministically with justification, or retain and flag. Preserve original raw files. Do not silently drop data or fill unknown business values with zero.

**Required validations:** unique property/unit/lease/work-order/reading IDs; valid `property_id`→properties and `unit_id`→units; start ≤ end for dated leases; opened ≤ closed for resolved work orders; `priority` ∈ {CRITICAL,HIGH,MEDIUM,LOW}; valid status; numeric nonnegative `energy_kwh`, `cost_usd`, and `monthly_rent_usd`; positive `floor_area_sqm`; `reading_month` must fall inside Jan–Jun 2026; at most one reading per `(unit_id, reading_month)`.

**Quarantine contract:** produce a file per dataset containing original row, `reason_code`, and `run_id`. Report input, accepted, quarantined, and duplicate counts, with an explicit reconciliation explaining whether exact duplicates are counted as quarantined. For aggregate KPI computation, a work order missing an SLA response can be retained for volume reporting but must not count as SLA-compliant by default. If you choose another defensible treatment, document it.

**Evidence:** `src/quality.py`, `data/quarantine/`, reconciliation CSV/JSON, at least five assertions or unit tests including foreign-key and duplicate checks.

In [19]:
# To confirm all datasets were loaded successfully, print the number of rows in each DataFrame.
print("Datasets available:")

for name, df in datasets.items():
    print(f"{name}: {len(df):,} rows")

Datasets available:
properties.csv: 12 rows
units.csv: 241 rows
leases.csv: 198 rows
work_orders.csv: 1,501 rows
meter_readings.csv: 1,440 rows


In [ ]:
# STUDENT TASK 2: write quality profiling and validation code in this cell or src/quality.py.
# Suggested starter pattern (expand for all five sources):
import pandas as pd

def profile_table(df: pd.DataFrame, identifier: str) -> dict:
    # TODO: return row count, nulls by column, duplicate identifier count, and types.
    raise NotImplementedError('Student implementation required')

def validate_and_quarantine(datasets: dict, run_id: str):
    # TODO: parse dates and numbers, enforce source and FK rules, reconcile rejects.
    # Return: clean_datasets, quarantine_datasets, quality_summary
    raise NotImplementedError('Student implementation required')

# TODO: tests for duplicate work order, unknown unit, negative kWh, missing response and bad date.
# TODO: persist clean records into a staging area, separate from untouched raw data.

### Solving Topic 2

In [20]:
def profile_table(df: pd.DataFrame, identifier: str) -> dict:
    """
    Produce basic quality statistics for a source table.
    """

    return {
        "row_count": len(df),

        "column_count": len(df.columns),

        "null_counts": df.isna().sum().to_dict(),

        "duplicate_identifier_count": (
            df[identifier].duplicated(keep=False).sum()
            if identifier in df.columns
            else None
        ),

        "duplicate_rows": int(df.duplicated().sum()),

        "data_types": {
            column: str(dtype)
            for column, dtype in df.dtypes.items()
        }
    }

#### Define the primary identifier of each dataset

In [21]:
IDENTIFIERS = {
    "properties.csv": "property_id",
    "units.csv": "unit_id",
    "leases.csv": "lease_id",
    "work_orders.csv": "work_order_id",
    "meter_readings.csv": "reading_id"
}

#### Profile all Five Datasets

In [22]:
profiles = {}

for filename, identifier in IDENTIFIERS.items():

    profiles[filename] = profile_table(
        datasets[filename],
        identifier
    )

    print("=" * 70)
    print(filename)
    print("=" * 70)

    print(
        "Rows:",
        profiles[filename]["row_count"]
    )

    print(
        "Duplicate identifiers:",
        profiles[filename]["duplicate_identifier_count"]
    )

    print(
        "Exact duplicate rows:",
        profiles[filename]["duplicate_rows"]
    )

    print("\nNull counts:")

    for column, count in profiles[filename]["null_counts"].items():
        print(f"  {column}: {count}")

    print()

properties.csv
Rows: 12
Duplicate identifiers: 0
Exact duplicate rows: 0

Null counts:
  property_id: 0
  property_name: 0
  city: 0

units.csv
Rows: 241
Duplicate identifiers: 2
Exact duplicate rows: 1

Null counts:
  unit_id: 0
  property_id: 0
  floor_area_sqm: 0
  active: 0

leases.csv
Rows: 198
Duplicate identifiers: 0
Exact duplicate rows: 0

Null counts:
  lease_id: 0
  unit_id: 0
  start_date: 0
  end_date: 139
  monthly_rent_usd: 0

work_orders.csv
Rows: 1501
Duplicate identifiers: 2
Exact duplicate rows: 1

Null counts:
  work_order_id: 0
  unit_id: 0
  opened_date: 0
  closed_date: 295
  priority: 0
  response_hours: 295
  cost_usd: 0
  status: 0

meter_readings.csv
Rows: 1440
Duplicate identifiers: 0
Exact duplicate rows: 0

Null counts:
  reading_id: 0
  unit_id: 0
  reading_month: 0
  energy_kwh: 0



#### Display and Inspect the actual duplicated records

In [23]:
# Units with duplicate identifiers
units_df = datasets["units.csv"]

units_df[
    units_df["unit_id"].duplicated(keep=False)
].sort_values("unit_id")

,unit_id,property_id,floor_area_sqm,active
4,P001-U005,P001,150,1
240,P001-U005,P001,150,1


In [24]:
# Work Orders
work_orders_df = datasets["work_orders.csv"]

work_orders_df[
    work_orders_df["work_order_id"].duplicated(keep=False)
].sort_values("work_order_id")

,work_order_id,unit_id,opened_date,closed_date,priority,response_hours,cost_usd,status
8,WO000009,P008-U016,2026-02-27,2026-03-02,MEDIUM,69.7,153.97,RESOLVED
1500,WO000009,P008-U016,2026-02-27,2026-03-02,MEDIUM,69.7,153.97,RESOLVED


#### Inspect all exact duplicate rows

In [25]:
for filename, df in datasets.items():

    duplicate_rows = df[df.duplicated(keep=False)]

    print(
        f"{filename}: "
        f"{len(duplicate_rows)} rows participating in exact duplicates"
    )

properties.csv: 0 rows participating in exact duplicates
units.csv: 2 rows participating in exact duplicates
leases.csv: 0 rows participating in exact duplicates
work_orders.csv: 2 rows participating in exact duplicates
meter_readings.csv: 0 rows participating in exact duplicates


#### Inspect Pandas data types 

In [26]:
for filename, df in datasets.items():

    print("\n", "=" * 60)
    print(filename)
    print("=" * 60)

    print(df.dtypes)


properties.csv
property_id      object
property_name    object
city             object
dtype: object

units.csv
unit_id           object
property_id       object
floor_area_sqm     int64
active             int64
dtype: object

leases.csv
lease_id            object
unit_id             object
start_date          object
end_date            object
monthly_rent_usd     int64
dtype: object

work_orders.csv
work_order_id      object
unit_id            object
opened_date        object
closed_date        object
priority           object
response_hours    float64
cost_usd          float64
status             object
dtype: object

meter_readings.csv
reading_id        object
unit_id           object
reading_month     object
energy_kwh       float64
dtype: object


#### compact quality-profile table

In [27]:
profile_summary = []

for filename, identifier in IDENTIFIERS.items():

    df = datasets[filename]

    profile_summary.append({
        "dataset": filename,
        "rows": len(df),
        "columns": len(df.columns),
        "null_cells": int(df.isna().sum().sum()),
        "duplicate_identifier_rows":
            int(df[identifier].duplicated(keep=False).sum()),
        "exact_duplicate_rows":
            int(df.duplicated(keep=False).sum())
    })

profile_summary_df = pd.DataFrame(profile_summary)

profile_summary_df

,dataset,rows,columns,null_cells,duplicate_identifier_rows,exact_duplicate_rows
0,properties.csv,12,3,0,0,0
1,units.csv,241,4,0,2,2
2,leases.csv,198,5,139,0,0
3,work_orders.csv,1501,8,590,2,2
4,meter_readings.csv,1440,4,0,0,0


#### Create a run ID

In [28]:
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

print("Quality run ID:", RUN_ID)

Quality run ID: 20261015T095317Z


#### Normalize Data for Validation

In [29]:
clean_working = {
    name: df.copy(deep=True)
    for name, df in datasets.items()
}

properties = clean_working["properties.csv"]
units = clean_working["units.csv"]
leases = clean_working["leases.csv"]
work_orders = clean_working["work_orders.csv"]
meter_readings = clean_working["meter_readings.csv"]

print("Working copies created.")

Working copies created.


#### Convert numeric columns

In [30]:
units["floor_area_sqm"] = pd.to_numeric(
    units["floor_area_sqm"],
    errors="coerce"
)

units["active"] = pd.to_numeric(
    units["active"],
    errors="coerce"
)

leases["monthly_rent_usd"] = pd.to_numeric(
    leases["monthly_rent_usd"],
    errors="coerce"
)

work_orders["response_hours"] = pd.to_numeric(
    work_orders["response_hours"],
    errors="coerce"
)

work_orders["cost_usd"] = pd.to_numeric(
    work_orders["cost_usd"],
    errors="coerce"
)

meter_readings["energy_kwh"] = pd.to_numeric(
    meter_readings["energy_kwh"],
    errors="coerce"
)

#### Parse all Dates

In [31]:
leases["start_date"] = pd.to_datetime(
    leases["start_date"],
    errors="coerce"
)

leases["end_date"] = pd.to_datetime(
    leases["end_date"],
    errors="coerce"
)

work_orders["opened_date"] = pd.to_datetime(
    work_orders["opened_date"],
    errors="coerce"
)

work_orders["closed_date"] = pd.to_datetime(
    work_orders["closed_date"],
    errors="coerce"
)

meter_readings["reading_month"] = pd.to_datetime(
    meter_readings["reading_month"],
    errors="coerce"
)

#### Detect duplicate identifiers

In [32]:
duplicate_unit_mask = units["unit_id"].duplicated(
    keep="first"
)

duplicate_work_order_mask = work_orders[
    "work_order_id"
].duplicated(
    keep="first"
)

duplicate_lease_mask = leases[
    "lease_id"
].duplicated(
    keep="first"
)

duplicate_reading_mask = meter_readings[
    "reading_id"
].duplicated(
    keep="first"
)

duplicate_property_mask = properties[
    "property_id"
].duplicated(
    keep="first"
)

In [33]:
# Inspect 
print("Duplicate properties:", duplicate_property_mask.sum())
print("Duplicate units:", duplicate_unit_mask.sum())
print("Duplicate leases:", duplicate_lease_mask.sum())
print("Duplicate work orders:", duplicate_work_order_mask.sum())
print("Duplicate readings:", duplicate_reading_mask.sum()) 


Duplicate properties: 0
Duplicate units: 1
Duplicate leases: 0
Duplicate work orders: 1
Duplicate readings: 0


#### Detect broken foreign keys

In [34]:
valid_property_ids = set(properties["property_id"])

invalid_unit_property_mask = ~units[
    "property_id"
].isin(valid_property_ids)

print(
    "Units with unknown property:",
    invalid_unit_property_mask.sum()
)

Units with unknown property: 0


#### Leases must reference valid units

In [35]:
valid_unit_ids = set(
    units.loc[
        ~duplicate_unit_mask,
        "unit_id"
    ]
)

invalid_lease_unit_mask = ~leases[
    "unit_id"
].isin(valid_unit_ids)

print(
    "Leases with unknown unit:",
    invalid_lease_unit_mask.sum()
)

Leases with unknown unit: 0


In [36]:
# Work Orders
invalid_work_order_unit_mask = ~work_orders[
    "unit_id"
].isin(valid_unit_ids)

print(
    "Work orders with unknown unit:",
    invalid_work_order_unit_mask.sum()
)

Work orders with unknown unit: 1


In [37]:
# Meter Readings
invalid_reading_unit_mask = ~meter_readings[
    "unit_id"
].isin(valid_unit_ids)

print(
    "Meter readings with unknown unit:",
    invalid_reading_unit_mask.sum()
)

Meter readings with unknown unit: 1


#### Detect invalid numeric values

In [38]:
invalid_floor_area_mask = (
    units["floor_area_sqm"].isna()
    | (units["floor_area_sqm"] <= 0)
)

negative_rent_mask = (
    leases["monthly_rent_usd"].isna()
    | (leases["monthly_rent_usd"] < 0)
)

negative_cost_mask = (
    work_orders["cost_usd"].isna()
    | (work_orders["cost_usd"] < 0)
)

negative_energy_mask = (
    meter_readings["energy_kwh"].isna()
    | (meter_readings["energy_kwh"] < 0)
)

In [39]:
print("Invalid floor areas:", invalid_floor_area_mask.sum())
print("Invalid rents:", negative_rent_mask.sum())
print("Invalid work-order costs:", negative_cost_mask.sum())
print("Invalid energy readings:", negative_energy_mask.sum())

Invalid floor areas: 0
Invalid rents: 0
Invalid work-order costs: 1
Invalid energy readings: 1


#### Validate categories

In [40]:
VALID_PRIORITIES = {
    "CRITICAL",
    "HIGH",
    "MEDIUM",
    "LOW"
}

invalid_priority_mask = ~work_orders[
    "priority"
].isin(VALID_PRIORITIES)

In [41]:
# For Status
VALID_STATUSES = {
    "OPEN",
    "RESOLVED"
}

invalid_status_mask = ~work_orders[
    "status"
].isin(VALID_STATUSES)

In [42]:
# Check  
print(
    "Invalid priorities:",
    invalid_priority_mask.sum()
)

print(
    "Invalid statuses:",
    invalid_status_mask.sum()
)

Invalid priorities: 0
Invalid statuses: 0


#### Validate date chronology

In [43]:
invalid_lease_dates_mask = (
    leases["start_date"].isna()
    |
    (
        leases["end_date"].notna()
        & (
            leases["start_date"]
            > leases["end_date"]
        )
    )
)

In [44]:
# Resolved Work Orders
invalid_work_order_dates_mask = (
    (work_orders["status"] == "RESOLVED")
    &
    (
        work_orders["opened_date"].isna()
        |
        work_orders["closed_date"].isna()
        |
        (
            work_orders["opened_date"]
            > work_orders["closed_date"]
        )
    )
)

In [45]:
# Check 
print(
    "Invalid lease dates:",
    invalid_lease_dates_mask.sum()
)

print(
    "Invalid work-order dates:",
    invalid_work_order_dates_mask.sum()
)

Invalid lease dates: 0
Invalid work-order dates: 0


#### Validate the meter-reading period
- The Project only permits:
  - January 2026 through June 2026

In [46]:
# Creating Boundaries for the report period. This will be used to filter the data for the report generation.
REPORT_START = pd.Timestamp("2026-01-01")
REPORT_END = pd.Timestamp("2026-06-30")

In [47]:
invalid_reading_month_mask = (
    meter_readings["reading_month"].isna()
    |
    (meter_readings["reading_month"] < REPORT_START)
    |
    (meter_readings["reading_month"] > REPORT_END)
)

In [48]:
# Check 
print(
    "Invalid reading months:",
    invalid_reading_month_mask.sum()
)

Invalid reading months: 0


### Build Clean and Quarantine Datasets

In [49]:
STAGING_DIR.mkdir(parents=True, exist_ok=True)
QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)

print("Staging directory:", STAGING_DIR)
print("Quarantine directory:", QUARANTINE_DIR)

Staging directory: c:\DataEngineeringProjects\proptech360-project\data\staging
Quarantine directory: c:\DataEngineeringProjects\proptech360-project\data\quarantine


In [50]:
def build_quarantine(df, mask, reason_code, run_id):
    """
    Return the rows matching mask with reason_code and run_id attached.
    """
    rejected = df.loc[mask].copy()

    rejected["reason_code"] = reason_code
    rejected["run_id"] = run_id

    return rejected

In [51]:
build_quarantine(
    units,
    duplicate_unit_mask,
    "DUPLICATE_UNIT_ID",
    RUN_ID
)

,unit_id,property_id,floor_area_sqm,active,reason_code,run_id
240,P001-U005,P001,150,1,DUPLICATE_UNIT_ID,20261015T095317Z


#### Quarantine duplicate units


In [52]:
units_quarantine = build_quarantine(
    units,
    duplicate_unit_mask,
    "DUPLICATE_UNIT_ID",
    RUN_ID
)

In [53]:
units_quarantine

,unit_id,property_id,floor_area_sqm,active,reason_code,run_id
240,P001-U005,P001,150,1,DUPLICATE_UNIT_ID,20261015T095317Z


In [54]:
units_clean = units.loc[
    ~duplicate_unit_mask
].copy()

In [55]:
print("Input units:", len(units))
print("Clean units:", len(units_clean))
print("Quarantined units:", len(units_quarantine))

Input units: 241
Clean units: 240
Quarantined units: 1


In [56]:
assert len(units) == len(units_clean) + len(units_quarantine)

In [57]:
properties_clean = properties.copy()

properties_quarantine = pd.DataFrame(
    columns=list(properties.columns)
    + ["reason_code", "run_id"]
)

In [58]:
print("Properties accepted:", len(properties_clean))
print("Properties quarantined:", len(properties_quarantine))

Properties accepted: 12
Properties quarantined: 0


#### Clean Leases

In [59]:
lease_reject_mask = (
    duplicate_lease_mask
    | invalid_lease_unit_mask
    | negative_rent_mask
    | invalid_lease_dates_mask
)

In [60]:
leases_quarantine_parts = []

for mask, reason in [
    (duplicate_lease_mask, "DUPLICATE_LEASE_ID"),
    (invalid_lease_unit_mask, "UNKNOWN_UNIT"),
    (negative_rent_mask, "INVALID_MONTHLY_RENT"),
    (invalid_lease_dates_mask, "INVALID_LEASE_DATES"),
]:
    q = build_quarantine(
        leases,
        mask,
        reason,
        RUN_ID
    )

    if not q.empty:
        leases_quarantine_parts.append(q)

In [61]:
if leases_quarantine_parts:
    leases_quarantine = pd.concat(
        leases_quarantine_parts,
        ignore_index=True
    )
else:
    leases_quarantine = pd.DataFrame(
        columns=list(leases.columns)
        + ["reason_code", "run_id"]
    )

In [62]:
leases_clean = leases.loc[
    ~lease_reject_mask
].copy()

In [63]:
print("Input leases:", len(leases))
print("Accepted leases:", len(leases_clean))
print("Quarantined leases:", lease_reject_mask.sum())

Input leases: 198
Accepted leases: 198
Quarantined leases: 0


#### Combined Mask

In [64]:
work_order_reject_mask = (
    duplicate_work_order_mask
    | invalid_work_order_unit_mask
    | negative_cost_mask
    | invalid_priority_mask
    | invalid_status_mask
    | invalid_work_order_dates_mask
)

#### Quarantine Pieces

In [65]:
work_order_quarantine_parts = []

work_order_rules = [
    (
        duplicate_work_order_mask,
        "DUPLICATE_WORK_ORDER_ID"
    ),
    (
        invalid_work_order_unit_mask,
        "UNKNOWN_UNIT"
    ),
    (
        negative_cost_mask,
        "INVALID_NEGATIVE_COST"
    ),
    (
        invalid_priority_mask,
        "INVALID_PRIORITY"
    ),
    (
        invalid_status_mask,
        "INVALID_STATUS"
    ),
    (
        invalid_work_order_dates_mask,
        "INVALID_WORK_ORDER_DATES"
    )
]

for mask, reason in work_order_rules:
    q = build_quarantine(
        work_orders,
        mask,
        reason,
        RUN_ID
    )

    if not q.empty:
        work_order_quarantine_parts.append(q)

In [66]:
if work_order_quarantine_parts:
    work_orders_quarantine = pd.concat(
        work_order_quarantine_parts,
        ignore_index=True
    )
else:
    work_orders_quarantine = pd.DataFrame(
        columns=list(work_orders.columns)
        + ["reason_code", "run_id"]
    )

In [67]:
# Create Clean Records
work_orders_clean = work_orders.loc[
    ~work_order_reject_mask
].copy()

In [68]:
# Retain SLA flag
work_orders_clean["sla_response_missing"] = (
    (work_orders_clean["status"] == "RESOLVED")
    & work_orders_clean["response_hours"].isna()
)

In [69]:
if not work_orders_quarantine.empty:
    work_orders_quarantine = (
        work_orders_quarantine
        .drop_duplicates(
            subset=["work_order_id"],
            keep="first"
        )
        .reset_index(drop=True)
    )

In [72]:
duplicate_unit_month_mask = meter_readings.duplicated(
    subset=["unit_id", "reading_month"],
    keep="first"
)

print(
    "Duplicate unit-month readings:",
    duplicate_unit_month_mask.sum()
)

Duplicate unit-month readings: 0


In [73]:
reading_reject_mask = (
    duplicate_reading_mask
    | invalid_reading_unit_mask
    | negative_energy_mask
    | invalid_reading_month_mask
    | duplicate_unit_month_mask
)

In [74]:
for name in [
    "duplicate_reading_mask",
    "invalid_reading_unit_mask",
    "negative_energy_mask",
    "invalid_reading_month_mask",
    "duplicate_unit_month_mask"
]:
    print(name, "exists" if name in globals() else "MISSING")

duplicate_reading_mask exists
invalid_reading_unit_mask exists
negative_energy_mask exists
invalid_reading_month_mask exists
duplicate_unit_month_mask exists


In [75]:
reading_quarantine_parts = []

reading_rules = [
    (
        duplicate_reading_mask,
        "DUPLICATE_READING_ID"
    ),
    (
        invalid_reading_unit_mask,
        "UNKNOWN_UNIT"
    ),
    (
        negative_energy_mask,
        "INVALID_NEGATIVE_ENERGY"
    ),
    (
        invalid_reading_month_mask,
        "INVALID_READING_MONTH"
    ),
    (
        duplicate_unit_month_mask,
        "DUPLICATE_UNIT_MONTH"
    )
]

for mask, reason in reading_rules:
    q = build_quarantine(
        meter_readings,
        mask,
        reason,
        RUN_ID
    )

    if not q.empty:
        reading_quarantine_parts.append(q)

In [76]:
if reading_quarantine_parts:
    meter_readings_quarantine = pd.concat(
        reading_quarantine_parts,
        ignore_index=True
    )
else:
    meter_readings_quarantine = pd.DataFrame(
        columns=list(meter_readings.columns)
        + ["reason_code", "run_id"]
    )

#### Remove duplicate quarantine representation:

In [ ]:
if not meter_readings_quarantine.empty:
    meter_readings_quarantine = (
        meter_readings_quarantine
        .drop_duplicates(
            subset=["reading_id"],
            keep="first"
        )
        .reset_index(drop=True)
    )

In [77]:
meter_readings_clean = meter_readings.loc[
    ~reading_reject_mask
].copy()

#### Inspect the quarantine records

In [78]:
print("Units quarantine:")
display(units_quarantine)

print("Work orders quarantine:")
display(work_orders_quarantine)

print("Meter readings quarantine:")
display(meter_readings_quarantine)

Units quarantine:


,unit_id,property_id,floor_area_sqm,active,reason_code,run_id
240,P001-U005,P001,150,1,DUPLICATE_UNIT_ID,20261015T095317Z


Work orders quarantine:


,work_order_id,unit_id,opened_date,closed_date,priority,response_hours,cost_usd,status,reason_code,run_id
0,WO000009,P008-U016,2026-02-27,2026-03-02,MEDIUM,69.7,153.97,RESOLVED,DUPLICATE_WORK_ORDER_ID,20261015T095317Z
1,WO000035,P999-U999,2026-04-08,NaT,LOW,NaN,82.75,OPEN,UNKNOWN_UNIT,20261015T095317Z
2,WO000064,P003-U019,2026-06-05,2026-06-12,MEDIUM,8.1,-300.00,RESOLVED,INVALID_NEGATIVE_COST,20261015T095317Z


Meter readings quarantine:


,reading_id,unit_id,reading_month,energy_kwh,reason_code,run_id
0,R000035,P999-U999,2026-05-01,1205.75,UNKNOWN_UNIT,20261015T095317Z
1,R000018,P001-U003,2026-06-01,-45.00,INVALID_NEGATIVE_ENERGY,20261015T095317Z


#### Save clean staging datasets

In [79]:
clean_datasets = {
    "properties.csv": properties_clean,
    "units.csv": units_clean,
    "leases.csv": leases_clean,
    "work_orders.csv": work_orders_clean,
    "meter_readings.csv": meter_readings_clean,
}

#### Save

In [80]:
for filename, df in clean_datasets.items():
    output_path = STAGING_DIR / filename

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved {filename}: "
        f"{len(df):,} rows"
    )

Saved properties.csv: 12 rows
Saved units.csv: 240 rows
Saved leases.csv: 198 rows
Saved work_orders.csv: 1,498 rows
Saved meter_readings.csv: 1,438 rows


#### Save quarantine files

In [81]:
quarantine_datasets = {
    "properties_quarantine.csv":
        properties_quarantine,

    "units_quarantine.csv":
        units_quarantine,

    "leases_quarantine.csv":
        leases_quarantine,

    "work_orders_quarantine.csv":
        work_orders_quarantine,

    "meter_readings_quarantine.csv":
        meter_readings_quarantine,
}

In [82]:
for filename, df in quarantine_datasets.items():
    output_path = QUARANTINE_DIR / filename

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved {filename}: "
        f"{len(df):,} rows"
    )

Saved properties_quarantine.csv: 0 rows
Saved units_quarantine.csv: 1 rows
Saved leases_quarantine.csv: 0 rows
Saved work_orders_quarantine.csv: 3 rows
Saved meter_readings_quarantine.csv: 2 rows


#### Reconciliation report

In [83]:
quality_summary = pd.DataFrame([
    {
        "dataset": "properties.csv",
        "input_rows": len(properties),
        "accepted_rows": len(properties_clean),
        "quarantined_rows":
            len(properties_quarantine),
        "duplicate_rows":
            int(
                duplicate_property_mask.sum()
            )
    },
    {
        "dataset": "units.csv",
        "input_rows": len(units),
        "accepted_rows": len(units_clean),
        "quarantined_rows":
            len(units_quarantine),
        "duplicate_rows":
            int(
                duplicate_unit_mask.sum()
            )
    },
    {
        "dataset": "leases.csv",
        "input_rows": len(leases),
        "accepted_rows": len(leases_clean),
        "quarantined_rows":
            int(lease_reject_mask.sum()),
        "duplicate_rows":
            int(
                duplicate_lease_mask.sum()
            )
    },
    {
        "dataset": "work_orders.csv",
        "input_rows": len(work_orders),
        "accepted_rows":
            len(work_orders_clean),
        "quarantined_rows":
            int(work_order_reject_mask.sum()),
        "duplicate_rows":
            int(
                duplicate_work_order_mask.sum()
            )
    },
    {
        "dataset": "meter_readings.csv",
        "input_rows": len(meter_readings),
        "accepted_rows":
            len(meter_readings_clean),
        "quarantined_rows":
            int(reading_reject_mask.sum()),
        "duplicate_rows":
            int(
                duplicate_reading_mask.sum()
            )
    }
])

quality_summary

,dataset,input_rows,accepted_rows,quarantined_rows,duplicate_rows
0,properties.csv,12,12,0,0
1,units.csv,241,240,1,1
2,leases.csv,198,198,0,0
3,work_orders.csv,1501,1498,3,1
4,meter_readings.csv,1440,1438,2,0


#### Assert Reconciliation

In [84]:
quality_summary["reconciled"] = (
    quality_summary["input_rows"]
    ==
    quality_summary["accepted_rows"]
    +
    quality_summary["quarantined_rows"]
)

quality_summary

,dataset,input_rows,accepted_rows,quarantined_rows,duplicate_rows,reconciled
0,properties.csv,12,12,0,0,True
1,units.csv,241,240,1,1,True
2,leases.csv,198,198,0,0,True
3,work_orders.csv,1501,1498,3,1,True
4,meter_readings.csv,1440,1438,2,0,True


In [85]:
assert quality_summary[
    "reconciled"
].all(), "Quality reconciliation failed"

In [89]:
missing_resolved_response_mask = (
    (work_orders["status"] == "RESOLVED")
    & (work_orders["response_hours"].isna())
)

print(
    "Resolved work orders missing response:",
    int(missing_resolved_response_mask.sum())
)

Resolved work orders missing response: 0


In [90]:
rule_summary = pd.DataFrame([
    {
        "rule": "duplicate_unit_id",
        "violations": int(duplicate_unit_mask.sum())
    },
    {
        "rule": "duplicate_work_order_id",
        "violations": int(duplicate_work_order_mask.sum())
    },
    {
        "rule": "unknown_work_order_unit",
        "violations": int(invalid_work_order_unit_mask.sum())
    },
    {
        "rule": "unknown_reading_unit",
        "violations": int(invalid_reading_unit_mask.sum())
    },
    {
        "rule": "negative_cost",
        "violations": int(negative_cost_mask.sum())
    },
    {
        "rule": "negative_energy",
        "violations": int(negative_energy_mask.sum())
    },
    {
        "rule": "missing_resolved_response",
        "violations": int(missing_resolved_response_mask.sum())
    },
    {
        "rule": "invalid_lease_dates",
        "violations": int(invalid_lease_dates_mask.sum())
    },
    {
        "rule": "invalid_work_order_dates",
        "violations": int(invalid_work_order_dates_mask.sum())
    },
    {
        "rule": "invalid_priority",
        "violations": int(invalid_priority_mask.sum())
    },
    {
        "rule": "invalid_status",
        "violations": int(invalid_status_mask.sum())
    },
    {
        "rule": "invalid_reading_month",
        "violations": int(invalid_reading_month_mask.sum())
    },
    {
        "rule": "duplicate_unit_month_reading",
        "violations": int(duplicate_unit_month_mask.sum())
    }
])

rule_summary

,rule,violations
0,duplicate_unit_id,1
1,duplicate_work_order_id,1
2,unknown_work_order_unit,1
3,unknown_reading_unit,1
4,negative_cost,1
5,negative_energy,1
6,missing_resolved_response,0
7,invalid_lease_dates,0
8,invalid_work_order_dates,0
9,invalid_priority,0


In [91]:
rule_summary_path = (
    CURATED_DIR / "quality_rule_summary.csv"
)

rule_summary.to_csv(
    rule_summary_path,
    index=False
)

print(
    "Rule summary saved to:",
    rule_summary_path
)

Rule summary saved to: c:\DataEngineeringProjects\proptech360-project\data\curated\quality_rule_summary.csv


#### Save the reconciliation report

In [86]:
quality_summary_path = (
    CURATED_DIR
    / "data_quality_summary.csv"
)

quality_summary.to_csv(
    quality_summary_path,
    index=False
)

print(
    "Quality summary saved to:",
    quality_summary_path
)

Quality summary saved to: c:\DataEngineeringProjects\proptech360-project\data\curated\data_quality_summary.csv


In [92]:
required_masks = [
    "duplicate_unit_mask",
    "duplicate_work_order_mask",
    "invalid_work_order_unit_mask",
    "invalid_reading_unit_mask",
    "negative_cost_mask",
    "negative_energy_mask",
    "missing_resolved_response_mask",
    "invalid_lease_dates_mask",
    "invalid_work_order_dates_mask",
    "invalid_priority_mask",
    "invalid_status_mask",
    "invalid_reading_month_mask",
    "duplicate_unit_month_mask",
]

for name in required_masks:
    print(
        f"{name}:",
        "OK" if name in globals() else "MISSING"
    )

duplicate_unit_mask: OK
duplicate_work_order_mask: OK
invalid_work_order_unit_mask: OK
invalid_reading_unit_mask: OK
negative_cost_mask: OK
negative_energy_mask: OK
missing_resolved_response_mask: OK
invalid_lease_dates_mask: OK
invalid_work_order_dates_mask: OK
invalid_priority_mask: OK
invalid_status_mask: OK
invalid_reading_month_mask: OK
duplicate_unit_month_mask: OK


In [93]:
rule_summary.to_csv(
    CURATED_DIR / "quality_rule_summary.csv",
    index=False
)

### Detect Overlapping Leases

In [94]:
lease_overlap_check = leases_clean.copy()

lease_overlap_check["effective_end_date"] = (
    lease_overlap_check["end_date"]
    .fillna(pd.Timestamp("2099-12-31"))
)

lease_overlap_check = lease_overlap_check.sort_values(
    by=["unit_id", "start_date", "effective_end_date"]
).reset_index(drop=True)

In [ ]:
# create previous-lease information within each unit
lease_overlap_check["previous_end_date"] = (
    lease_overlap_check
    .groupby("unit_id")["effective_end_date"]
    .shift(1)
)

lease_overlap_check["previous_lease_id"] = (
    lease_overlap_check
    .groupby("unit_id")["lease_id"]
    .shift(1)
)

In [ ]:
# Detect Overlap
lease_overlap_check["overlap_flag"] = (
    lease_overlap_check["previous_end_date"].notna()
    &
    (
        lease_overlap_check["start_date"]
        <= lease_overlap_check["previous_end_date"]
    )
)

In [97]:
# Inspect the results
overlapping_leases = lease_overlap_check.loc[
    lease_overlap_check["overlap_flag"],
    [
        "unit_id",
        "previous_lease_id",
        "lease_id",
        "start_date",
        "end_date",
        "previous_end_date"
    ]
].copy()

print("Overlapping lease records:", len(overlapping_leases))

overlapping_leases

Overlapping lease records: 0


,unit_id,previous_lease_id,lease_id,start_date,end_date,previous_end_date


#### Adding overlap to the quality-rule summary

In [98]:
overlap_summary_row = pd.DataFrame([
    {
        "rule": "overlapping_leases",
        "violations": int(len(overlapping_leases))
    }
])

rule_summary = pd.concat(
    [rule_summary, overlap_summary_row],
    ignore_index=True
)

rule_summary

,rule,violations
0,duplicate_unit_id,1
1,duplicate_work_order_id,1
2,unknown_work_order_unit,1
3,unknown_reading_unit,1
4,negative_cost,1
5,negative_energy,1
6,missing_resolved_response,0
7,invalid_lease_dates,0
8,invalid_work_order_dates,0
9,invalid_priority,0


#### Save Updated File

In [99]:
rule_summary.to_csv(
    CURATED_DIR / "quality_rule_summary.csv",
    index=False
)

### Automated Data Quality Assertions

In [100]:
assert properties_clean["property_id"].is_unique, \
    "Test failed: property_id is not unique"

print("PASS: property IDs are unique")

PASS: property IDs are unique


In [101]:
assert units_clean["unit_id"].is_unique, \
    "Test failed: unit_id is not unique"

print("PASS: unit IDs are unique")

PASS: unit IDs are unique


In [102]:
assert work_orders_clean["work_order_id"].is_unique, \
    "Test failed: work_order_id is not unique"

print("PASS: work-order IDs are unique")

PASS: work-order IDs are unique


In [103]:
assert units_clean["property_id"].isin(
    properties_clean["property_id"]
).all(), \
    "Test failed: units contain unknown property_id values"

print("PASS: all unit property foreign keys are valid")

PASS: all unit property foreign keys are valid


In [104]:
assert leases_clean["unit_id"].isin(
    units_clean["unit_id"]
).all(), \
    "Test failed: leases contain unknown unit_id values"

print("PASS: all lease unit foreign keys are valid")

PASS: all lease unit foreign keys are valid


In [105]:
assert work_orders_clean["unit_id"].isin(
    units_clean["unit_id"]
).all(), \
    "Test failed: work orders contain unknown unit_id values"

print("PASS: all work-order unit foreign keys are valid")

PASS: all work-order unit foreign keys are valid


In [106]:
assert meter_readings_clean["unit_id"].isin(
    units_clean["unit_id"]
).all(), \
    "Test failed: meter readings contain unknown unit_id values"

print("PASS: all meter-reading unit foreign keys are valid")

PASS: all meter-reading unit foreign keys are valid


In [107]:
assert (
    meter_readings_clean["energy_kwh"] >= 0
).all(), \
    "Test failed: negative energy values remain"

print("PASS: energy values are nonnegative")

PASS: energy values are nonnegative


In [108]:
assert (
    work_orders_clean["cost_usd"] >= 0
).all(), \
    "Test failed: negative work-order costs remain"

print("PASS: work-order costs are nonnegative")

PASS: work-order costs are nonnegative


In [109]:
assert (
    units_clean["floor_area_sqm"] > 0
).all(), \
    "Test failed: non-positive floor area remains"

print("PASS: floor areas are positive")

PASS: floor areas are positive


In [110]:
assert meter_readings_clean["reading_month"].between(
    REPORT_START,
    REPORT_END
).all(), \
    "Test failed: reading_month outside reporting window"

print("PASS: all readings fall within Jan-Jun 2026")

PASS: all readings fall within Jan-Jun 2026


In [111]:
assert not meter_readings_clean.duplicated(
    subset=["unit_id", "reading_month"]
).any(), \
    "Test failed: duplicate unit-month meter reading exists"

print("PASS: unit-month meter readings are unique")

PASS: unit-month meter readings are unique


In [112]:
valid_lease_order = (
    leases_clean["end_date"].isna()
    |
    (
        leases_clean["start_date"]
        <= leases_clean["end_date"]
    )
)

assert valid_lease_order.all(), \
    "Test failed: invalid lease chronology remains"

print("PASS: lease date chronology is valid")

PASS: lease date chronology is valid


In [113]:
resolved_orders = work_orders_clean[
    work_orders_clean["status"] == "RESOLVED"
]

assert (
    resolved_orders["opened_date"]
    <= resolved_orders["closed_date"]
).all(), \
    "Test failed: resolved work order closes before it opens"

print("PASS: resolved work-order chronology is valid")

PASS: resolved work-order chronology is valid


In [114]:
assert len(overlapping_leases) == 0, \
    "Test failed: overlapping leases remain"

print("PASS: no overlapping leases detected")

PASS: no overlapping leases detected


In [115]:
test_work_orders = pd.DataFrame([
    {
        "work_order_id": "TEST001",
        "unit_id": "P001-U001"
    },
    {
        "work_order_id": "TEST001",
        "unit_id": "P001-U001"
    }
])

assert test_work_orders[
    "work_order_id"
].duplicated().sum() == 1

print("PASS: duplicate work-order test")

PASS: duplicate work-order test


In [116]:
test_unit_id = "P999-U999"

assert test_unit_id not in set(
    units_clean["unit_id"]
)

print("PASS: unknown-unit test")

PASS: unknown-unit test


In [117]:
test_energy = pd.Series([500, -45, 300])

assert (test_energy < 0).sum() == 1

print("PASS: negative-energy test")

PASS: negative-energy test


In [118]:
test_missing_response = pd.DataFrame({
    "status": ["RESOLVED", "OPEN"],
    "response_hours": [None, None]
})

test_mask = (
    (test_missing_response["status"] == "RESOLVED")
    &
    test_missing_response["response_hours"].isna()
)

assert test_mask.sum() == 1

print("PASS: missing resolved-response test")

PASS: missing resolved-response test


In [119]:
test_dates = pd.DataFrame({
    "start_date": pd.to_datetime(
        ["2026-06-01"]
    ),
    "end_date": pd.to_datetime(
        ["2026-05-01"]
    )
})

bad_date_mask = (
    test_dates["start_date"]
    > test_dates["end_date"]
)

assert bad_date_mask.sum() == 1

print("PASS: invalid-date test")

PASS: invalid-date test


In [120]:
print("=" * 60)
print("TOPIC 2 AUTOMATED QUALITY TESTS PASSED")
print("=" * 60)

TOPIC 2 AUTOMATED QUALITY TESTS PASSED


## 7. Topic 3 — PySpark curated transformations (25 points)

**Tasks:** (a) start a SparkSession; load validated staging data using explicit schemas where practical; (b) generate the six calendar month ends; (c) compute a unit-month occupancy and monthly-rent snapshot without duplicate lease multiplication; (d) aggregate maintenance per property-month with compliant and noncompliant **resolved** cases plus open cases; (e) aggregate valid energy by property-month; (f) join the independently aggregated KPI tables on `(property_id, month)`; (g) write partitioned Parquet by `month` to `data/curated/property_month_kpis/`.

**Critical transformations:** use DataFrame joins, groupBy/aggregations, casts, date operations, `when`, and window functions where appropriate. Avoid collecting entire fact tables to the driver. Verify all 12 properties × 6 months = **72 property-month output rows** provided no property has been intentionally excluded. Use `countDistinct` or a pre-aggregated unit-month snapshot to avoid multiplying occupancy and rent during joins.

**Minimum columns:** `property_id`, `month`, `total_units`, `occupied_units`, `occupancy_rate`, `earned_monthly_rent_usd`, `resolved_orders`, `sla_eligible_orders`, `sla_compliant_orders`, `open_orders`, `sla_compliance_rate`, `energy_kwh`, `floor_area_sqm`, `energy_intensity_kwh_sqm`.

**Edge cases:** no leases at month end, all orders unresolved, missing response durations, zero/invalid floor area, and duplicate identifiers. Explain your chosen denominator for each rate. Write a quality assertion for unique `(property_id, month)` and a range check for percentages.

In [4]:
import os
import sys

# Explicitly tell Spark to use this notebook's Python interpreter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = (
    SparkSession.builder
    .appName("PropTech360-Project")
    .master("local[1]")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Python executable:", sys.executable)

Spark version: 4.2.0
Python executable: c:\DataEngineeringProjects\proptech360-project\venv\Scripts\python.exe


### Load Validated Staging Data

In [5]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType,
    BooleanType
)

In [6]:
# Properties schema for PySpark DataFrame
properties_schema = StructType([
    StructField("property_id", StringType(), False),
    StructField("property_name", StringType(), False),
    StructField("city", StringType(), False),
])

In [7]:
# Units schema for PySpark DataFrame
units_schema = StructType([
    StructField("unit_id", StringType(), False),
    StructField("property_id", StringType(), False),
    StructField("floor_area_sqm", DoubleType(), False),
    StructField("active", IntegerType(), False),
])

In [8]:
# Leases schema for PySpark DataFrame
leases_schema = StructType([
    StructField("lease_id", StringType(), False),
    StructField("unit_id", StringType(), False),
    StructField("start_date", DateType(), False),
    StructField("end_date", DateType(), True),
    StructField("monthly_rent_usd", DoubleType(), False),
])

In [9]:
# work_orders schema for PySpark DataFrame
work_orders_schema = StructType([
    StructField("work_order_id", StringType(), False),
    StructField("unit_id", StringType(), False),
    StructField("opened_date", DateType(), False),
    StructField("closed_date", DateType(), True),
    StructField("priority", StringType(), False),
    StructField("response_hours", DoubleType(), True),
    StructField("cost_usd", DoubleType(), False),
    StructField("status", StringType(), False),
    StructField("sla_response_missing", BooleanType(), True),
])

In [10]:
# Meter readings schema for PySpark DataFrame
meter_readings_schema = StructType([
    StructField("reading_id", StringType(), False),
    StructField("unit_id", StringType(), False),
    StructField("reading_month", DateType(), False),
    StructField("energy_kwh", DoubleType(), False),
])

#### Reading The Staging Files

In [11]:
STAGING_SPARK_PATH = STAGING_DIR.as_posix()

print(STAGING_SPARK_PATH)

c:/DataEngineeringProjects/proptech360-project/data/staging


In [12]:
properties_sdf = (
    spark.read
    .option("header", True)
    .schema(properties_schema)
    .csv(f"{STAGING_SPARK_PATH}/properties.csv")
)

units_sdf = (
    spark.read
    .option("header", True)
    .schema(units_schema)
    .csv(f"{STAGING_SPARK_PATH}/units.csv")
)

leases_sdf = (
    spark.read
    .option("header", True)
    .schema(leases_schema)
    .csv(f"{STAGING_SPARK_PATH}/leases.csv")
)

work_orders_sdf = (
    spark.read
    .option("header", True)
    .schema(work_orders_schema)
    .csv(f"{STAGING_SPARK_PATH}/work_orders.csv")
)

meter_readings_sdf = (
    spark.read
    .option("header", True)
    .schema(meter_readings_schema)
    .csv(f"{STAGING_SPARK_PATH}/meter_readings.csv")
)

In [13]:
print("Properties:", properties_sdf.count())
print("Units:", units_sdf.count())
print("Leases:", leases_sdf.count())
print("Work orders:", work_orders_sdf.count())
print("Meter readings:", meter_readings_sdf.count())

Properties: 12
Units: 240
Leases: 198
Work orders: 1498
Meter readings: 1438


#### Inspect the Spark schemas

In [14]:
properties_sdf.printSchema()
units_sdf.printSchema()
leases_sdf.printSchema()
work_orders_sdf.printSchema()
meter_readings_sdf.printSchema()

root
 |-- property_id: string (nullable = true)
 |-- property_name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- unit_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- active: integer (nullable = true)

root
 |-- lease_id: string (nullable = true)
 |-- unit_id: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- monthly_rent_usd: double (nullable = true)

root
 |-- work_order_id: string (nullable = true)
 |-- unit_id: string (nullable = true)
 |-- opened_date: date (nullable = true)
 |-- closed_date: date (nullable = true)
 |-- priority: string (nullable = true)
 |-- response_hours: double (nullable = true)
 |-- cost_usd: double (nullable = true)
 |-- status: string (nullable = true)
 |-- sla_response_missing: boolean (nullable = true)

root
 |-- reading_id: string (nullable = true)
 |-- unit_id: string (nullable = true)
 |-- readin

In [15]:
properties_sdf.show(5, truncate=False)

+-----------+-------------------+-------------+
|property_id|property_name      |city         |
+-----------+-------------------+-------------+
|P001       |MetroSpace Tower 01|Lagos        |
|P002       |MetroSpace Tower 02|Abuja        |
|P003       |MetroSpace Tower 03|Ibadan       |
|P004       |MetroSpace Tower 04|Port Harcourt|
|P005       |MetroSpace Tower 05|Lagos        |
+-----------+-------------------+-------------+
only showing top 5 rows


In [16]:
units_sdf.show(5, truncate=False)

+---------+-----------+--------------+------+
|unit_id  |property_id|floor_area_sqm|active|
+---------+-----------+--------------+------+
|P001-U001|P001       |75.0          |1     |
|P001-U002|P001       |90.0          |1     |
|P001-U003|P001       |75.0          |1     |
|P001-U004|P001       |90.0          |1     |
|P001-U005|P001       |150.0         |1     |
+---------+-----------+--------------+------+
only showing top 5 rows


In [17]:
leases_sdf.show(5, truncate=False)

+--------+---------+----------+----------+----------------+
|lease_id|unit_id  |start_date|end_date  |monthly_rent_usd|
+--------+---------+----------+----------+----------------+
|L00001  |P001-U001|2025-03-07|NULL      |975.0           |
|L00003  |P001-U003|2025-08-17|NULL      |975.0           |
|L00004  |P001-U004|2025-07-03|2026-12-26|800.0           |
|L00005  |P001-U005|2025-03-16|NULL      |975.0           |
|L00006  |P001-U006|2025-01-02|NULL      |800.0           |
+--------+---------+----------+----------+----------------+
only showing top 5 rows


In [18]:
work_orders_sdf.show(5, truncate=False)

+-------------+---------+-----------+-----------+--------+--------------+--------+--------+--------------------+
|work_order_id|unit_id  |opened_date|closed_date|priority|response_hours|cost_usd|status  |sla_response_missing|
+-------------+---------+-----------+-----------+--------+--------------+--------+--------+--------------------+
|WO000001     |P009-U003|2026-01-24 |NULL       |HIGH    |NULL          |103.66  |OPEN    |false               |
|WO000002     |P002-U005|2026-04-09 |2026-04-21 |CRITICAL|47.4          |989.1   |RESOLVED|false               |
|WO000003     |P001-U020|2026-01-28 |2026-01-28 |MEDIUM  |15.5          |1154.72 |RESOLVED|false               |
|WO000004     |P010-U011|2026-06-06 |2026-06-09 |CRITICAL|43.0          |402.84  |RESOLVED|false               |
|WO000005     |P004-U001|2026-06-20 |2026-06-21 |MEDIUM  |33.3          |316.66  |RESOLVED|false               |
+-------------+---------+-----------+-----------+--------+--------------+--------+--------+-----

In [19]:
meter_readings_sdf.show(5, truncate=False)


+----------+---------+-------------+----------+
|reading_id|unit_id  |reading_month|energy_kwh|
+----------+---------+-------------+----------+
|R000001   |P001-U001|2026-01-01   |631.99    |
|R000002   |P001-U001|2026-02-01   |1237.66   |
|R000003   |P001-U001|2026-03-01   |1024.61   |
|R000004   |P001-U001|2026-04-01   |942.58    |
|R000005   |P001-U001|2026-05-01   |1077.28   |
+----------+---------+-------------+----------+
only showing top 5 rows


#### six-month reporting calendar

In [20]:
calendar_data = [
    ("2026-01-31",),
    ("2026-02-28",),
    ("2026-03-31",),
    ("2026-04-30",),
    ("2026-05-31",),
    ("2026-06-30",),
]

calendar_sdf = spark.createDataFrame(
    calendar_data,
    ["month"]
)

In [21]:
calendar_sdf = calendar_sdf.withColumn(
    "month",
    F.to_date("month")
)

In [22]:
calendar_sdf.show()
calendar_sdf.printSchema()

+----------+
|     month|
+----------+
|2026-01-31|
|2026-02-28|
|2026-03-31|
|2026-04-30|
|2026-05-31|
|2026-06-30|
+----------+

root
 |-- month: date (nullable = true)



#### Create the 72-row property-month spine

In [23]:
property_month_spine = properties_sdf.select(
    "property_id"
).crossJoin(
    calendar_sdf
)

In [24]:
print(
    "Property-month rows:",
    property_month_spine.count()
)

Property-month rows: 72


In [25]:
property_month_spine.orderBy(
    "property_id",
    "month"
).show(15)

+-----------+----------+
|property_id|     month|
+-----------+----------+
|       P001|2026-01-31|
|       P001|2026-02-28|
|       P001|2026-03-31|
|       P001|2026-04-30|
|       P001|2026-05-31|
|       P001|2026-06-30|
|       P002|2026-01-31|
|       P002|2026-02-28|
|       P002|2026-03-31|
|       P002|2026-04-30|
|       P002|2026-05-31|
|       P002|2026-06-30|
|       P003|2026-01-31|
|       P003|2026-02-28|
|       P003|2026-03-31|
+-----------+----------+
only showing top 15 rows


#### Total active units and floor area

In [26]:
# Filter Active Units
active_units_sdf = units_sdf.filter(
    F.col("active") == 1
)

In [27]:
property_unit_summary = (
    active_units_sdf
    .groupBy("property_id")
    .agg(
        F.countDistinct("unit_id").alias(
            "total_units"
        ),
        F.sum("floor_area_sqm").alias(
            "floor_area_sqm"
        )
    )
)

In [28]:
property_unit_summary.orderBy(
    "property_id"
).show(12)

+-----------+-----------+--------------+
|property_id|total_units|floor_area_sqm|
+-----------+-----------+--------------+
|       P001|         20|        2040.0|
|       P002|         20|        1650.0|
|       P003|         20|        1995.0|
|       P004|         20|        1920.0|
|       P005|         20|        1860.0|
|       P006|         20|        1755.0|
|       P007|         20|        1785.0|
|       P008|         20|        1800.0|
|       P009|         20|        1695.0|
|       P010|         20|        1980.0|
|       P011|         20|        1575.0|
|       P012|         20|        1830.0|
+-----------+-----------+--------------+



#### Joining the unit denominator to all six months

In [29]:
base_property_month = (
    property_month_spine
    .join(
        property_unit_summary,
        on="property_id",
        how="left"
    )
)

In [30]:
base_property_month.orderBy(
    "property_id",
    "month"
).show(12)

+-----------+----------+-----------+--------------+
|property_id|     month|total_units|floor_area_sqm|
+-----------+----------+-----------+--------------+
|       P001|2026-01-31|         20|        2040.0|
|       P001|2026-02-28|         20|        2040.0|
|       P001|2026-03-31|         20|        2040.0|
|       P001|2026-04-30|         20|        2040.0|
|       P001|2026-05-31|         20|        2040.0|
|       P001|2026-06-30|         20|        2040.0|
|       P002|2026-01-31|         20|        1650.0|
|       P002|2026-02-28|         20|        1650.0|
|       P002|2026-03-31|         20|        1650.0|
|       P002|2026-04-30|         20|        1650.0|
|       P002|2026-05-31|         20|        1650.0|
|       P002|2026-06-30|         20|        1650.0|
+-----------+----------+-----------+--------------+
only showing top 12 rows


In [31]:
assert base_property_month.count() == 72

print(
    "PASS: base property-month grain contains 72 rows"
)

PASS: base property-month grain contains 72 rows


### unit-month occupancy and rent snapshot

In [32]:
leases_with_property = (
    leases_sdf
    .join(
        active_units_sdf.select(
            "unit_id",
            "property_id"
        ),
        on="unit_id",
        how="inner"
    )
)

In [33]:
lease_month_candidates = (
    leases_with_property
    .crossJoin(calendar_sdf)
)

In [34]:
active_lease_month = (
    lease_month_candidates
    .filter(
        (F.col("start_date") <= F.col("month"))
        &
        (
            F.col("end_date").isNull()
            |
            (F.col("end_date") >= F.col("month"))
        )
    )
)

In [35]:
active_lease_month.select(
    "lease_id",
    "unit_id",
    "property_id",
    "month",
    "monthly_rent_usd"
).orderBy(
    "property_id",
    "month",
    "unit_id"
).show(20)

+--------+---------+-----------+----------+----------------+
|lease_id|  unit_id|property_id|     month|monthly_rent_usd|
+--------+---------+-----------+----------+----------------+
|  L00001|P001-U001|       P001|2026-01-31|           975.0|
|  L00003|P001-U003|       P001|2026-01-31|           975.0|
|  L00004|P001-U004|       P001|2026-01-31|           800.0|
|  L00005|P001-U005|       P001|2026-01-31|           975.0|
|  L00006|P001-U006|       P001|2026-01-31|           800.0|
|  L00007|P001-U007|       P001|2026-01-31|           650.0|
|  L00008|P001-U008|       P001|2026-01-31|           800.0|
|  L00009|P001-U009|       P001|2026-01-31|           800.0|
|  L00010|P001-U010|       P001|2026-01-31|          1500.0|
|  L00011|P001-U011|       P001|2026-01-31|          1500.0|
|  L00012|P001-U012|       P001|2026-01-31|           800.0|
|  L00014|P001-U014|       P001|2026-01-31|           975.0|
|  L00015|P001-U015|       P001|2026-01-31|          2200.0|
|  L00017|P001-U017|    

#### unit-month grain

In [36]:
unit_month_lease_counts = (
    active_lease_month
    .groupBy(
        "unit_id",
        "month"
    )
    .agg(
        F.countDistinct("lease_id").alias(
            "active_lease_count"
        )
    )
)

In [37]:
# Check for conflicts
conflicting_unit_months = (
    unit_month_lease_counts
    .filter(
        F.col("active_lease_count") > 1
    )
)

print(
    "Conflicting unit-months:",
    conflicting_unit_months.count()
)

Conflicting unit-months: 0


In [38]:
assert conflicting_unit_months.count() == 0

print("PASS: no unit-month has multiple active leases")

PASS: no unit-month has multiple active leases


#### Aggregate occupancy and rent

In [39]:
occupancy_rent_kpi = (
    active_lease_month
    .groupBy(
        "property_id",
        "month"
    )
    .agg(
        F.countDistinct("unit_id").alias(
            "occupied_units"
        ),
        F.sum("monthly_rent_usd").alias(
            "earned_monthly_rent_usd"
        )
    )
)

In [40]:
occupancy_rent_kpi = (
    base_property_month
    .join(
        occupancy_rent_kpi,
        on=["property_id", "month"],
        how="left"
    )
)

In [ ]:
# Replace missing occupancy/rent with zero
occupancy_rent_kpi = (
    occupancy_rent_kpi
    .fillna({
        "occupied_units": 0,
        "earned_monthly_rent_usd": 0.0
    })
)

In [42]:
# Calculate occupancy rate 
occupancy_rent_kpi = (
    occupancy_rent_kpi
    .withColumn(
        "occupancy_rate",
        F.when(
            F.col("total_units") > 0,
            F.col("occupied_units")
            / F.col("total_units")
        )
    )
)

In [43]:
occupancy_rent_kpi.select(
    "property_id",
    "month",
    "total_units",
    "occupied_units",
    "occupancy_rate",
    "earned_monthly_rent_usd"
).orderBy(
    "property_id",
    "month"
).show(20)

+-----------+----------+-----------+--------------+--------------+-----------------------+
|property_id|     month|total_units|occupied_units|occupancy_rate|earned_monthly_rent_usd|
+-----------+----------+-----------+--------------+--------------+-----------------------+
|       P001|2026-01-31|         20|            17|          0.85|                20550.0|
|       P001|2026-02-28|         20|            17|          0.85|                20550.0|
|       P001|2026-03-31|         20|            17|          0.85|                20550.0|
|       P001|2026-04-30|         20|            17|          0.85|                20550.0|
|       P001|2026-05-31|         20|            17|          0.85|                20550.0|
|       P001|2026-06-30|         20|            16|           0.8|                19050.0|
|       P002|2026-01-31|         20|            18|           0.9|                22875.0|
|       P002|2026-02-28|         20|            18|           0.9|                22875.0|

#### Validate Occupancy 

In [44]:
assert occupancy_rent_kpi.count() == 72

In [45]:
invalid_occupancy = (
    occupancy_rent_kpi
    .filter(
        (F.col("occupancy_rate") < 0)
        |
        (F.col("occupancy_rate") > 1)
    )
    .count()
)

assert invalid_occupancy == 0

print("PASS: occupancy KPI contains 72 valid rows")

PASS: occupancy KPI contains 72 valid rows


#### Maintenance SLA metrics

In [46]:
work_orders_with_property = (
    work_orders_sdf
    .join(
        units_sdf.select(
            "unit_id",
            "property_id"
        ),
        on="unit_id",
        how="inner"
    )
)

In [47]:
work_orders_with_property = (
    work_orders_with_property
    .withColumn(
        "month",
        F.last_day("opened_date")
    )
)

In [48]:
work_orders_reporting = (
    work_orders_with_property
    .filter(
        (F.col("month") >= F.to_date(F.lit("2026-01-31")))
        &
        (F.col("month") <= F.to_date(F.lit("2026-06-30")))
    )
)

#### SLA thresholds

In [49]:
work_orders_reporting = (
    work_orders_reporting
    .withColumn(
        "sla_target_hours",
        F.when(
            F.col("priority") == "CRITICAL",
            F.lit(4.0)
        )
        .when(
            F.col("priority") == "HIGH",
            F.lit(12.0)
        )
        .when(
            F.col("priority") == "MEDIUM",
            F.lit(48.0)
        )
        .when(
            F.col("priority") == "LOW",
            F.lit(72.0)
        )
    )
)

In [50]:
work_orders_reporting = (
    work_orders_reporting
    .withColumn(
        "sla_eligible",
        F.when(
            (F.col("status") == "RESOLVED")
            &
            F.col("response_hours").isNotNull(),
            1
        ).otherwise(0)
    )
)

In [51]:
work_orders_reporting = (
    work_orders_reporting
    .withColumn(
        "sla_compliant",
        F.when(
            (F.col("sla_eligible") == 1)
            &
            (
                F.col("response_hours")
                <= F.col("sla_target_hours")
            ),
            1
        ).otherwise(0)
    )
)

In [52]:
work_orders_reporting = (
    work_orders_reporting
    .withColumn(
        "open_order",
        F.when(
            F.col("status") == "OPEN",
            1
        ).otherwise(0)
    )
)

#### Aggregate maintenance by property-month

In [53]:
maintenance_kpi = (
    work_orders_reporting
    .groupBy(
        "property_id",
        "month"
    )
    .agg(
        F.sum(
            F.when(
                F.col("status") == "RESOLVED",
                1
            ).otherwise(0)
        ).alias("resolved_orders"),

        F.sum("sla_eligible").alias(
            "sla_eligible_orders"
        ),

        F.sum("sla_compliant").alias(
            "sla_compliant_orders"
        ),

        F.sum("open_order").alias(
            "open_orders"
        )
    )
)

In [54]:
# Compliance Rate Calculation
maintenance_kpi = (
    maintenance_kpi
    .withColumn(
        "sla_compliance_rate",
        F.when(
            F.col("sla_eligible_orders") > 0,
            F.col("sla_compliant_orders")
            / F.col("sla_eligible_orders")
        )
    )
)

In [55]:
maintenance_kpi.orderBy(
    "property_id",
    "month"
).show(20)

+-----------+----------+---------------+-------------------+--------------------+-----------+-------------------+
|property_id|     month|resolved_orders|sla_eligible_orders|sla_compliant_orders|open_orders|sla_compliance_rate|
+-----------+----------+---------------+-------------------+--------------------+-----------+-------------------+
|       P001|2026-01-31|             14|                 14|                   4|          2| 0.2857142857142857|
|       P001|2026-02-28|             18|                 18|                   7|          2| 0.3888888888888889|
|       P001|2026-03-31|             13|                 13|                   3|          4|0.23076923076923078|
|       P001|2026-04-30|             11|                 11|                   4|          4|0.36363636363636365|
|       P001|2026-05-31|             16|                 16|                   7|          8|             0.4375|
|       P001|2026-06-30|             13|                 13|                   5|       

#### Build Energy Metrics

In [56]:
energy_with_property = (
    meter_readings_sdf
    .join(
        units_sdf.select(
            "unit_id",
            "property_id"
        ),
        on="unit_id",
        how="inner"
    )
)

In [57]:
energy_with_property = (
    energy_with_property
    .withColumn(
        "month",
        F.last_day("reading_month")
    )
)

In [58]:
# Aggregate 
energy_kpi = (
    energy_with_property
    .groupBy(
        "property_id",
        "month"
    )
    .agg(
        F.sum("energy_kwh").alias(
            "energy_kwh"
        )
    )
)

In [59]:
energy_kpi.orderBy(
    "property_id",
    "month"
).show(20)

+-----------+----------+------------------+
|property_id|     month|        energy_kwh|
+-----------+----------+------------------+
|       P001|2026-01-31|14426.600000000006|
|       P001|2026-02-28|15429.519999999999|
|       P001|2026-03-31|12825.479999999996|
|       P001|2026-04-30|          14927.42|
|       P001|2026-05-31|13203.629999999997|
|       P001|2026-06-30|10576.650000000001|
|       P002|2026-01-31|           12268.4|
|       P002|2026-02-28|          13003.27|
|       P002|2026-03-31|14047.880000000005|
|       P002|2026-04-30|12719.989999999998|
|       P002|2026-05-31|15514.630000000001|
|       P002|2026-06-30|14097.849999999999|
|       P003|2026-01-31|14856.789999999999|
|       P003|2026-02-28|          12852.64|
|       P003|2026-03-31|          13852.46|
|       P003|2026-04-30|10622.919999999998|
|       P003|2026-05-31|          11382.21|
|       P003|2026-06-30|13610.589999999998|
|       P004|2026-01-31|          15588.96|
|       P004|2026-02-28|13702.67

#### Joining the three KPI families

In [60]:
final_kpi = occupancy_rent_kpi

final_kpi = (
    final_kpi
    .join(
        maintenance_kpi,
        on=["property_id", "month"],
        how="left"
    )
)

In [61]:
final_kpi = (
    final_kpi
    .join(
        energy_kpi,
        on=["property_id", "month"],
        how="left"
    )
)

In [62]:
final_kpi = (
    final_kpi
    .fillna({
        "resolved_orders": 0,
        "sla_eligible_orders": 0,
        "sla_compliant_orders": 0,
        "open_orders": 0,
        "energy_kwh": 0.0
    })
)

#### Energy Intensity Calculation

In [63]:
final_kpi = (
    final_kpi
    .withColumn(
        "energy_intensity_kwh_sqm",
        F.when(
            F.col("floor_area_sqm") > 0,
            F.col("energy_kwh")
            / F.col("floor_area_sqm")
        )
    )
)

#### Required Columns:

In [64]:
final_kpi = final_kpi.select(
    "property_id",
    "month",
    "total_units",
    "occupied_units",
    "occupancy_rate",
    "earned_monthly_rent_usd",
    "resolved_orders",
    "sla_eligible_orders",
    "sla_compliant_orders",
    "open_orders",
    "sla_compliance_rate",
    "energy_kwh",
    "floor_area_sqm",
    "energy_intensity_kwh_sqm"
)

In [65]:
final_kpi.orderBy(
    "property_id",
    "month"
).show(30, truncate=False)

+-----------+----------+-----------+--------------+--------------+-----------------------+---------------+-------------------+--------------------+-----------+-------------------+------------------+--------------+------------------------+
|property_id|month     |total_units|occupied_units|occupancy_rate|earned_monthly_rent_usd|resolved_orders|sla_eligible_orders|sla_compliant_orders|open_orders|sla_compliance_rate|energy_kwh        |floor_area_sqm|energy_intensity_kwh_sqm|
+-----------+----------+-----------+--------------+--------------+-----------------------+---------------+-------------------+--------------------+-----------+-------------------+------------------+--------------+------------------------+
|P001       |2026-01-31|20         |17            |0.85          |20550.0                |14             |14                 |4                   |2          |0.2857142857142857 |14426.600000000006|2040.0        |7.071862745098042       |
|P001       |2026-02-28|20         |17      

#### Validate Final Result

In [66]:
row_count = final_kpi.count()

print("Final KPI rows:", row_count)

assert row_count == 72

Final KPI rows: 72


#### Uniqueness:

In [67]:
duplicate_grain_count = (
    final_kpi
    .groupBy(
        "property_id",
        "month"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

assert duplicate_grain_count == 0

print("PASS: property-month grain is unique")

PASS: property-month grain is unique


#### Validate Occupancy:

In [68]:
assert (
    final_kpi
    .filter(
        (F.col("occupancy_rate") < 0)
        |
        (F.col("occupancy_rate") > 1)
    )
    .count()
    == 0
)

print("PASS: occupancy rates are valid")

PASS: occupancy rates are valid


#### Validate SLA:

In [69]:
assert (
    final_kpi
    .filter(
        F.col("sla_compliance_rate").isNotNull()
        &
        (
            (F.col("sla_compliance_rate") < 0)
            |
            (F.col("sla_compliance_rate") > 1)
        )
    )
    .count()
    == 0
)

print("PASS: SLA compliance rates are valid")

PASS: SLA compliance rates are valid


#### Validate Occupied Units

In [70]:
assert (
    final_kpi
    .filter(
        F.col("occupied_units")
        > F.col("total_units")
    )
    .count()
    == 0
)

print("PASS: occupied units never exceed total units")

PASS: occupied units never exceed total units


### Partitioned Parquet 

In [71]:
# Output Directory: 
CURATED_SPARK_PATH = (
    CURATED_DIR
    / "property_month_kpis"
).as_posix()

print(CURATED_SPARK_PATH)

c:/DataEngineeringProjects/proptech360-project/data/curated/property_month_kpis


In [73]:
hadoop_version = (
    spark._jvm
    .org.apache.hadoop.util.VersionInfo
    .getVersion()
)

print("Hadoop version:", hadoop_version)

Hadoop version: 3.5.0


In [74]:
import os

print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print(
    "hadoop.home.dir:",
    spark.sparkContext._jsc.hadoopConfiguration().get(
        "hadoop.home.dir"
    )
)

HADOOP_HOME: None
hadoop.home.dir: None


In [75]:
print("Final KPI rows:", final_kpi.count())

final_kpi.select(
    "property_id",
    "month",
    "occupancy_rate",
    "earned_monthly_rent_usd",
    "sla_compliance_rate",
    "energy_kwh"
).show(5)

Final KPI rows: 72
+-----------+----------+--------------+-----------------------+-------------------+------------------+
|property_id|     month|occupancy_rate|earned_monthly_rent_usd|sla_compliance_rate|        energy_kwh|
+-----------+----------+--------------+-----------------------+-------------------+------------------+
|       P001|2026-01-31|          0.85|                20550.0| 0.2857142857142857|14426.600000000006|
|       P002|2026-01-31|           0.9|                22875.0| 0.6111111111111112|           12268.4|
|       P003|2026-01-31|           0.9|                22850.0| 0.5333333333333333|14856.789999999999|
|       P004|2026-01-31|           0.8|                21050.0| 0.5263157894736842|          15588.96|
|       P005|2026-01-31|           0.8|                19200.0| 0.3157894736842105|13274.710000000001|
+-----------+----------+--------------+-----------------------+-------------------+------------------+
only showing top 5 rows


In [76]:
final_kpi_pd = final_kpi.toPandas()

print(final_kpi_pd.shape)

(72, 14)


In [77]:
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

In [78]:
PARQUET_OUTPUT = CURATED_DIR / "property_month_kpis"

PARQUET_OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

In [79]:
table = pa.Table.from_pandas(
    final_kpi_pd,
    preserve_index=False
)

pq.write_to_dataset(
    table,
    root_path=str(PARQUET_OUTPUT),
    partition_cols=["month"]
)

print("Partitioned Parquet written successfully.")

Partitioned Parquet written successfully.


In [80]:
for path in sorted(PARQUET_OUTPUT.iterdir()):
    print(path.name)

month=2026-01-31
month=2026-02-28
month=2026-03-31
month=2026-04-30
month=2026-05-31
month=2026-06-30


## 8. Topic 4 — PostgreSQL dimensional mart and analytics (20 points)

**Tasks:** Create a PostgreSQL database `proptech360` and dimensional schema with `dim_property`, `dim_month`, and `fact_property_month`. Document natural keys, surrogate keys if used, primary and foreign keys, NOT NULL constraints, valid-range checks, and indexes. Load the curated property-month dataset into the fact table using parameterized inserts or a bulk loader. Implement an idempotent `INSERT ... ON CONFLICT ... DO UPDATE` strategy at the `(property_id, month)` grain; rerun loading and prove row counts stay stable.

**Write and submit SQL answering:**

1. Rank properties by average monthly occupancy for January–June 2026, displaying numerator and denominator context.
2. Calculate monthly earned contractual rent by city (not payment receipts).
3. Show maintenance SLA compliance by property, including counts of eligible resolved and excluded unresolved/missing-response records.
4. Find property-months with energy intensity above the portfolio's monthly average; show both intensity and total kWh.
5. Create a portfolio dashboard view with occupancy rate, earned rent, SLA compliance, and energy intensity; explain handling of NULL rates.

**Evidence:** `sql/01_schema.sql`, `02_load.sql` or `src/load_postgres.py`, `03_analytics.sql`, and screenshots/text output for five queries. Verify PostgreSQL results reconcile with Spark aggregates. Do not paste database credentials into the notebook.

In [81]:
import os

from dotenv import load_dotenv
import psycopg

load_dotenv()

PG_HOST = os.getenv("PG_HOST")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

In [82]:
required_vars = {
    "PG_HOST": PG_HOST,
    "PG_DATABASE": PG_DATABASE,
    "PG_USER": PG_USER,
    "PG_PASSWORD": PG_PASSWORD,
}

missing = [
    name
    for name, value in required_vars.items()
    if not value
]

if missing:
    raise ValueError(
        f"Missing environment variables: {missing}"
    )

print("PostgreSQL configuration loaded successfully.")

PostgreSQL configuration loaded successfully.


In [83]:
conn = psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD,
)

print("Connected to PostgreSQL successfully.")

conn.close()

Connected to PostgreSQL successfully.


### Create PostgreSQL Dimensional Schema

In [84]:
from pathlib import Path

SCHEMA_PATH = PROJECT_ROOT / "sql" / "schema.sql"

print("Schema file:", SCHEMA_PATH)
print("Exists:", SCHEMA_PATH.exists())

Schema file: c:\DataEngineeringProjects\proptech360-project\sql\schema.sql
Exists: True


In [85]:
with open(SCHEMA_PATH, "r", encoding="utf-8") as file:
    schema_sql = file.read()

with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:
        cur.execute(schema_sql)

print("PostgreSQL dimensional schema created successfully.")

PostgreSQL dimensional schema created successfully.


In [86]:
# Verify the tables in the database
with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """)

        tables = cur.fetchall()

for table in tables:
    print(table[0])

dim_month
dim_property
fact_property_month


#### Prepare dim_property using the validated staging dataset

In [88]:
import pandas as pd

In [89]:
properties_for_db = pd.read_csv(
    STAGING_DIR / "properties.csv"
)

print(properties_for_db.shape)
properties_for_db.head()

(12, 3)


,property_id,property_name,city
0,P001,MetroSpace Tower 01,Lagos
1,P002,MetroSpace Tower 02,Abuja
2,P003,MetroSpace Tower 03,Ibadan
3,P004,MetroSpace Tower 04,Port Harcourt
4,P005,MetroSpace Tower 05,Lagos


In [90]:
# Validate that the property_id column is unique
assert len(properties_for_db) == 12
assert properties_for_db["property_id"].is_unique

print("PASS: property dimension source is valid")

PASS: property dimension source is valid


#### Prepare dim_month: 

In [91]:
months_for_db = (
    final_kpi_pd[["month"]]
    .drop_duplicates()
    .sort_values("month")
    .copy()
)

months_for_db["month"] = pd.to_datetime(
    months_for_db["month"]
)

months_for_db["year"] = (
    months_for_db["month"].dt.year
)

months_for_db["month_number"] = (
    months_for_db["month"].dt.month
)

months_for_db["month_name"] = (
    months_for_db["month"].dt.month_name()
)

months_for_db

,month,year,month_number,month_name
0,2026-01-31,2026,1,January
12,2026-02-28,2026,2,February
24,2026-03-31,2026,3,March
36,2026-04-30,2026,4,April
48,2026-05-31,2026,5,May
60,2026-06-30,2026,6,June


In [92]:
assert len(months_for_db) == 6

print("PASS: month dimension contains 6 months")

PASS: month dimension contains 6 months


#### Load dim_property idempotently

In [93]:
property_upsert_sql = """
INSERT INTO dim_property (
    property_id,
    property_name,
    city
)
VALUES (%s, %s, %s)

ON CONFLICT (property_id)
DO UPDATE SET
    property_name = EXCLUDED.property_name,
    city = EXCLUDED.city;
"""

In [94]:
with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:

        for row in properties_for_db.itertuples(
            index=False
        ):

            cur.execute(
                property_upsert_sql,
                (
                    row.property_id,
                    row.property_name,
                    row.city
                )
            )

print("dim_property loaded successfully.")

dim_property loaded successfully.


#### Load dim_month idempotently

In [95]:
month_upsert_sql = """
INSERT INTO dim_month (
    month_date,
    year,
    month_number,
    month_name
)
VALUES (%s, %s, %s, %s)

ON CONFLICT (month_date)
DO UPDATE SET
    year = EXCLUDED.year,
    month_number = EXCLUDED.month_number,
    month_name = EXCLUDED.month_name;
"""

In [96]:
with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:

        for row in months_for_db.itertuples(
            index=False
        ):

            cur.execute(
                month_upsert_sql,
                (
                    row.month.date(),
                    int(row.year),
                    int(row.month_number),
                    row.month_name
                )
            )

print("dim_month loaded successfully.")

dim_month loaded successfully.


#### Dimension Counts Verification 

In [97]:
with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:

        cur.execute(
            "SELECT COUNT(*) FROM dim_property;"
        )
        property_count = cur.fetchone()[0]

        cur.execute(
            "SELECT COUNT(*) FROM dim_month;"
        )
        month_count = cur.fetchone()[0]

print("dim_property rows:", property_count)
print("dim_month rows:", month_count)

dim_property rows: 12
dim_month rows: 6


### Load fact_property_month Idempotently

In [98]:
print("Source rows:", len(final_kpi_pd))

duplicate_source_rows = (
    final_kpi_pd
    .duplicated(
        subset=["property_id", "month"]
    )
    .sum()
)

print(
    "Duplicate property-month rows:",
    duplicate_source_rows
)

assert len(final_kpi_pd) == 72
assert duplicate_source_rows == 0

print("PASS: fact source contains 72 unique property-month rows")

Source rows: 72
Duplicate property-month rows: 0
PASS: fact source contains 72 unique property-month rows


In [99]:
import math
import pandas as pd

def nullable_float(value):
    if pd.isna(value):
        return None
    return float(value)

#### fact-table UPSERT SQL

In [100]:
fact_upsert_sql = """
INSERT INTO fact_property_month (
    property_key,
    month_key,
    total_units,
    occupied_units,
    occupancy_rate,
    earned_monthly_rent_usd,
    resolved_orders,
    sla_eligible_orders,
    sla_compliant_orders,
    open_orders,
    sla_compliance_rate,
    energy_kwh,
    floor_area_sqm,
    energy_intensity_kwh_sqm
)

SELECT
    dp.property_key,
    dm.month_key,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s

FROM dim_property AS dp
CROSS JOIN dim_month AS dm

WHERE dp.property_id = %s
  AND dm.month_date = %s

ON CONFLICT (
    property_key,
    month_key
)

DO UPDATE SET
    total_units = EXCLUDED.total_units,
    occupied_units = EXCLUDED.occupied_units,
    occupancy_rate = EXCLUDED.occupancy_rate,
    earned_monthly_rent_usd = EXCLUDED.earned_monthly_rent_usd,
    resolved_orders = EXCLUDED.resolved_orders,
    sla_eligible_orders = EXCLUDED.sla_eligible_orders,
    sla_compliant_orders = EXCLUDED.sla_compliant_orders,
    open_orders = EXCLUDED.open_orders,
    sla_compliance_rate = EXCLUDED.sla_compliance_rate,
    energy_kwh = EXCLUDED.energy_kwh,
    floor_area_sqm = EXCLUDED.floor_area_sqm,
    energy_intensity_kwh_sqm = EXCLUDED.energy_intensity_kwh_sqm;
"""

#### Reusable fact loader

In [101]:
def load_fact_property_month(df):
    rows_processed = 0

    with psycopg.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DATABASE,
        user=PG_USER,
        password=PG_PASSWORD
    ) as conn:

        with conn.cursor() as cur:

            for row in df.itertuples(index=False):

                month_date = pd.to_datetime(
                    row.month
                ).date()

                cur.execute(
                    fact_upsert_sql,
                    (
                        int(row.total_units),
                        int(row.occupied_units),
                        nullable_float(
                            row.occupancy_rate
                        ),
                        float(
                            row.earned_monthly_rent_usd
                        ),
                        int(row.resolved_orders),
                        int(row.sla_eligible_orders),
                        int(row.sla_compliant_orders),
                        int(row.open_orders),
                        nullable_float(
                            row.sla_compliance_rate
                        ),
                        float(row.energy_kwh),
                        float(row.floor_area_sqm),
                        nullable_float(
                            row.energy_intensity_kwh_sqm
                        ),
                        row.property_id,
                        month_date
                    )
                )

                rows_processed += 1

    return rows_processed

In [102]:
# First Load 
rows_processed = load_fact_property_month(
    final_kpi_pd
)

print(
    "Fact rows processed:",
    rows_processed
)

Fact rows processed: 72


#### Database Count:

In [103]:
with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:

        cur.execute(
            "SELECT COUNT(*) FROM fact_property_month;"
        )

        fact_count_first_load = cur.fetchone()[0]

print(
    "fact_property_month rows after first load:",
    fact_count_first_load
)

fact_property_month rows after first load: 72


In [104]:
assert fact_count_first_load == 72

print("PASS: first fact load contains exactly 72 rows")

PASS: first fact load contains exactly 72 rows


In [105]:
with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                dp.property_id,
                dp.property_name,
                dm.month_date,
                f.total_units,
                f.occupied_units,
                f.occupancy_rate,
                f.earned_monthly_rent_usd,
                f.sla_compliance_rate,
                f.energy_kwh,
                f.energy_intensity_kwh_sqm
            FROM fact_property_month AS f

            JOIN dim_property AS dp
                ON f.property_key = dp.property_key

            JOIN dim_month AS dm
                ON f.month_key = dm.month_key

            ORDER BY
                dp.property_id,
                dm.month_date

            LIMIT 10;
        """)

        sample_rows = cur.fetchall()

for row in sample_rows:
    print(row)

('P001', 'MetroSpace Tower 01', datetime.date(2026, 1, 31), 20, 17, Decimal('0.850000'), Decimal('20550.00'), Decimal('0.285714'), Decimal('14426.60'), Decimal('7.071863'))
('P001', 'MetroSpace Tower 01', datetime.date(2026, 2, 28), 20, 17, Decimal('0.850000'), Decimal('20550.00'), Decimal('0.388889'), Decimal('15429.52'), Decimal('7.563490'))
('P001', 'MetroSpace Tower 01', datetime.date(2026, 3, 31), 20, 17, Decimal('0.850000'), Decimal('20550.00'), Decimal('0.230769'), Decimal('12825.48'), Decimal('6.287000'))
('P001', 'MetroSpace Tower 01', datetime.date(2026, 4, 30), 20, 17, Decimal('0.850000'), Decimal('20550.00'), Decimal('0.363636'), Decimal('14927.42'), Decimal('7.317363'))
('P001', 'MetroSpace Tower 01', datetime.date(2026, 5, 31), 20, 17, Decimal('0.850000'), Decimal('20550.00'), Decimal('0.437500'), Decimal('13203.63'), Decimal('6.472368'))
('P001', 'MetroSpace Tower 01', datetime.date(2026, 6, 30), 20, 16, Decimal('0.800000'), Decimal('19050.00'), Decimal('0.384615'), Deci

In [106]:
with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                COUNT(*) AS fact_rows,
                COUNT(DISTINCT property_key) AS properties,
                COUNT(DISTINCT month_key) AS months
            FROM fact_property_month;
        """)

        verification = cur.fetchone()

print("Fact rows:", verification[0])
print("Distinct properties:", verification[1])
print("Distinct months:", verification[2])

Fact rows: 72
Distinct properties: 12
Distinct months: 6


#### Loader Key Test: 

In [107]:
rows_processed_second_run = (
    load_fact_property_month(
        final_kpi_pd
    )
)

print(
    "Rows processed during second load:",
    rows_processed_second_run
)

Rows processed during second load: 72


#### Table Count: 

In [108]:
with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:

        cur.execute(
            "SELECT COUNT(*) FROM fact_property_month;"
        )

        fact_count_second_load = (
            cur.fetchone()[0]
        )

print(
    "Fact rows after second load:",
    fact_count_second_load
)

Fact rows after second load: 72


#### formal idempotency assertion

In [109]:
assert fact_count_first_load == 72
assert fact_count_second_load == 72

assert (
    fact_count_first_load
    ==
    fact_count_second_load
)

print(
    "PASS: fact load is idempotent. "
    "Row count remained 72 after rerun."
)

PASS: fact load is idempotent. Row count remained 72 after rerun.


#### full dimensional mart

In [110]:
with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:

        cur.execute("""
            SELECT 'dim_property', COUNT(*)
            FROM dim_property

            UNION ALL

            SELECT 'dim_month', COUNT(*)
            FROM dim_month

            UNION ALL

            SELECT 'fact_property_month', COUNT(*)
            FROM fact_property_month;
        """)

        mart_counts = cur.fetchall()

for table_name, row_count in mart_counts:
    print(
        f"{table_name}: {row_count} rows"
    )

dim_property: 12 rows
dim_month: 6 rows
fact_property_month: 72 rows


In [111]:
query_1 = """
SELECT
    dp.property_id,
    dp.property_name,
    dp.city,
    SUM(f.occupied_units) AS occupied_unit_months,
    SUM(f.total_units) AS available_unit_months,

    ROUND(
        SUM(f.occupied_units)::NUMERIC
        / NULLIF(SUM(f.total_units), 0),
        4
    ) AS average_occupancy_rate,

    ROUND(
        100.0 * SUM(f.occupied_units)::NUMERIC
        / NULLIF(SUM(f.total_units), 0),
        2
    ) AS average_occupancy_pct

FROM fact_property_month AS f

JOIN dim_property AS dp
    ON f.property_key = dp.property_key

JOIN dim_month AS dm
    ON f.month_key = dm.month_key

WHERE dm.month_date
      BETWEEN DATE '2026-01-01'
          AND DATE '2026-06-30'

GROUP BY
    dp.property_id,
    dp.property_name,
    dp.city

ORDER BY
    average_occupancy_rate DESC,
    dp.property_id;
"""

In [112]:
with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    query_1_df = pd.read_sql_query(
        query_1,
        conn
    )

query_1_df

C:\Users\stvic\AppData\Local\Temp\ipykernel_28920\2937960525.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query_1_df = pd.read_sql_query(


,property_id,property_name,city,occupied_unit_months,available_unit_months,average_occupancy_rate,average_occupancy_pct
0,P011,MetroSpace Tower 11,Ibadan,108,120,0.9000,90.00
1,P002,MetroSpace Tower 02,Abuja,107,120,0.8917,89.17
2,P008,MetroSpace Tower 08,Port Harcourt,104,120,0.8667,86.67
3,P001,MetroSpace Tower 01,Lagos,101,120,0.8417,84.17
4,P006,MetroSpace Tower 06,Abuja,100,120,0.8333,83.33
5,P003,MetroSpace Tower 03,Ibadan,97,120,0.8083,80.83
6,P005,MetroSpace Tower 05,Lagos,95,120,0.7917,79.17
7,P004,MetroSpace Tower 04,Port Harcourt,89,120,0.7417,74.17
8,P009,MetroSpace Tower 09,Lagos,88,120,0.7333,73.33
9,P010,MetroSpace Tower 10,Abuja,83,120,0.6917,69.17


In [113]:
query_2 = """
SELECT
    dm.month_date,
    dp.city,

    ROUND(
        SUM(f.earned_monthly_rent_usd),
        2
    ) AS earned_contractual_rent_usd

FROM fact_property_month AS f

JOIN dim_property AS dp
    ON f.property_key = dp.property_key

JOIN dim_month AS dm
    ON f.month_key = dm.month_key

GROUP BY
    dm.month_date,
    dp.city

ORDER BY
    dm.month_date,
    dp.city;
"""

with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    query_2_df = pd.read_sql_query(
        query_2,
        conn
    )

query_2_df

C:\Users\stvic\AppData\Local\Temp\ipykernel_28920\2447386255.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query_2_df = pd.read_sql_query(


,month_date,city,earned_contractual_rent_usd
0,2026-01-31,Abuja,69175.0
1,2026-01-31,Ibadan,62275.0
2,2026-01-31,Lagos,58725.0
3,2026-01-31,Port Harcourt,64250.0
4,2026-02-28,Abuja,69175.0
5,2026-02-28,Ibadan,62275.0
6,2026-02-28,Lagos,58725.0
7,2026-02-28,Port Harcourt,64250.0
8,2026-03-31,Abuja,67375.0
9,2026-03-31,Ibadan,60675.0


In [114]:
query_3 = """
SELECT
    dp.property_id,
    dp.property_name,
    dp.city,

    SUM(f.resolved_orders) AS resolved_orders,

    SUM(f.sla_eligible_orders)
        AS sla_eligible_resolved_orders,

    SUM(f.sla_compliant_orders)
        AS sla_compliant_orders,

    SUM(
        f.resolved_orders
        - f.sla_eligible_orders
    ) AS excluded_resolved_orders,

    SUM(f.open_orders)
        AS unresolved_open_orders,

    ROUND(
        SUM(f.sla_compliant_orders)::NUMERIC
        / NULLIF(SUM(f.sla_eligible_orders), 0),
        4
    ) AS sla_compliance_rate,

    ROUND(
        100.0
        * SUM(f.sla_compliant_orders)::NUMERIC
        / NULLIF(SUM(f.sla_eligible_orders), 0),
        2
    ) AS sla_compliance_pct

FROM fact_property_month AS f

JOIN dim_property AS dp
    ON f.property_key = dp.property_key

GROUP BY
    dp.property_id,
    dp.property_name,
    dp.city

ORDER BY
    sla_compliance_rate DESC NULLS LAST,
    dp.property_id;
"""

with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    query_3_df = pd.read_sql_query(
        query_3,
        conn
    )

query_3_df

C:\Users\stvic\AppData\Local\Temp\ipykernel_28920\3760952594.py:59: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query_3_df = pd.read_sql_query(


,property_id,property_name,city,resolved_orders,sla_eligible_resolved_orders,sla_compliant_orders,excluded_resolved_orders,unresolved_open_orders,sla_compliance_rate,sla_compliance_pct
0,P002,MetroSpace Tower 02,Abuja,100,100,46,0,25,0.4600,46.00
1,P004,MetroSpace Tower 04,Port Harcourt,96,96,37,0,19,0.3854,38.54
2,P008,MetroSpace Tower 08,Port Harcourt,98,98,36,0,27,0.3673,36.73
3,P010,MetroSpace Tower 10,Abuja,104,104,38,0,28,0.3654,36.54
4,P012,MetroSpace Tower 12,Port Harcourt,96,96,35,0,24,0.3646,36.46
5,P005,MetroSpace Tower 05,Lagos,104,104,37,0,24,0.3558,35.58
6,P001,MetroSpace Tower 01,Lagos,85,85,30,0,26,0.3529,35.29
7,P003,MetroSpace Tower 03,Ibadan,101,101,35,0,14,0.3465,34.65
8,P006,MetroSpace Tower 06,Abuja,107,107,35,0,34,0.3271,32.71
9,P007,MetroSpace Tower 07,Ibadan,111,111,36,0,15,0.3243,32.43


In [115]:
query_4 = """
WITH energy_comparison AS (
    SELECT
        dp.property_id,
        dp.property_name,
        dp.city,
        dm.month_date,
        f.energy_kwh,
        f.energy_intensity_kwh_sqm,

        AVG(
            f.energy_intensity_kwh_sqm
        ) OVER (
            PARTITION BY dm.month_date
        ) AS portfolio_monthly_avg_intensity

    FROM fact_property_month AS f

    JOIN dim_property AS dp
        ON f.property_key = dp.property_key

    JOIN dim_month AS dm
        ON f.month_key = dm.month_key

    WHERE
        f.energy_intensity_kwh_sqm
        IS NOT NULL
)

SELECT
    property_id,
    property_name,
    city,
    month_date,

    ROUND(
        energy_kwh,
        2
    ) AS energy_kwh,

    ROUND(
        energy_intensity_kwh_sqm,
        4
    ) AS energy_intensity_kwh_sqm,

    ROUND(
        portfolio_monthly_avg_intensity,
        4
    ) AS portfolio_monthly_avg_intensity

FROM energy_comparison

WHERE
    energy_intensity_kwh_sqm
    > portfolio_monthly_avg_intensity

ORDER BY
    month_date,
    energy_intensity_kwh_sqm DESC;
"""

with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    query_4_df = pd.read_sql_query(
        query_4,
        conn
    )

query_4_df

C:\Users\stvic\AppData\Local\Temp\ipykernel_28920\3678054073.py:70: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query_4_df = pd.read_sql_query(


,property_id,property_name,city,month_date,energy_kwh,energy_intensity_kwh_sqm,portfolio_monthly_avg_intensity
0,P004,MetroSpace Tower 04,Port Harcourt,2026-01-31,15588.96,8.1193,7.3110
1,P012,MetroSpace Tower 12,Port Harcourt,2026-01-31,14324.86,7.8278,7.3110
2,P009,MetroSpace Tower 09,Lagos,2026-01-31,13197.17,7.7859,7.3110
3,P003,MetroSpace Tower 03,Ibadan,2026-01-31,14856.79,7.4470,7.3110
4,P002,MetroSpace Tower 02,Abuja,2026-01-31,12268.40,7.4354,7.3110
5,P007,MetroSpace Tower 07,Ibadan,2026-01-31,13126.46,7.3538,7.3110
6,P008,MetroSpace Tower 08,Port Harcourt,2026-02-28,16891.70,9.3843,7.7679
7,P011,MetroSpace Tower 11,Ibadan,2026-02-28,14259.49,9.0536,7.7679
8,P007,MetroSpace Tower 07,Ibadan,2026-02-28,14900.22,8.3475,7.7679
9,P006,MetroSpace Tower 06,Abuja,2026-02-28,14139.49,8.0567,7.7679


In [116]:
create_dashboard_view_sql = """
CREATE OR REPLACE VIEW vw_portfolio_dashboard AS

SELECT
    dp.property_id,
    dp.property_name,
    dp.city,
    dm.month_date,
    dm.year,
    dm.month_number,
    dm.month_name,

    f.total_units,
    f.occupied_units,
    f.occupancy_rate,
    f.earned_monthly_rent_usd,

    f.resolved_orders,
    f.sla_eligible_orders,
    f.sla_compliant_orders,
    f.open_orders,
    f.sla_compliance_rate,

    f.energy_kwh,
    f.floor_area_sqm,
    f.energy_intensity_kwh_sqm

FROM fact_property_month AS f

JOIN dim_property AS dp
    ON f.property_key = dp.property_key

JOIN dim_month AS dm
    ON f.month_key = dm.month_key;
"""

with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    with conn.cursor() as cur:
        cur.execute(
            create_dashboard_view_sql
        )

print(
    "Portfolio dashboard view created successfully."
)

Portfolio dashboard view created successfully.


In [117]:
query_5 = """
SELECT *
FROM vw_portfolio_dashboard
ORDER BY
    property_id,
    month_date;
"""

with psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD
) as conn:

    query_5_df = pd.read_sql_query(
        query_5,
        conn
    )

print(query_5_df.shape)
query_5_df.head(12)

(72, 19)


C:\Users\stvic\AppData\Local\Temp\ipykernel_28920\126270463.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query_5_df = pd.read_sql_query(


,property_id,property_name,city,month_date,year,month_number,month_name,total_units,occupied_units,occupancy_rate,earned_monthly_rent_usd,resolved_orders,sla_eligible_orders,sla_compliant_orders,open_orders,sla_compliance_rate,energy_kwh,floor_area_sqm,energy_intensity_kwh_sqm
0,P001,MetroSpace Tower 01,Lagos,2026-01-31,2026,1,January,20,17,0.85,20550.0,14,14,4,2,0.285714,14426.60,2040.0,7.071863
1,P001,MetroSpace Tower 01,Lagos,2026-02-28,2026,2,February,20,17,0.85,20550.0,18,18,7,2,0.388889,15429.52,2040.0,7.563490
2,P001,MetroSpace Tower 01,Lagos,2026-03-31,2026,3,March,20,17,0.85,20550.0,13,13,3,4,0.230769,12825.48,2040.0,6.287000
3,P001,MetroSpace Tower 01,Lagos,2026-04-30,2026,4,April,20,17,0.85,20550.0,11,11,4,4,0.363636,14927.42,2040.0,7.317363
4,P001,MetroSpace Tower 01,Lagos,2026-05-31,2026,5,May,20,17,0.85,20550.0,16,16,7,8,0.437500,13203.63,2040.0,6.472368
5,P001,MetroSpace Tower 01,Lagos,2026-06-30,2026,6,June,20,16,0.80,19050.0,13,13,5,6,0.384615,10576.65,2040.0,5.184632
6,P002,MetroSpace Tower 02,Abuja,2026-01-31,2026,1,January,20,18,0.90,22875.0,18,18,11,3,0.611111,12268.40,1650.0,7.435394
7,P002,MetroSpace Tower 02,Abuja,2026-02-28,2026,2,February,20,18,0.90,22875.0,13,13,3,7,0.230769,13003.27,1650.0,7.880770
8,P002,MetroSpace Tower 02,Abuja,2026-03-31,2026,3,March,20,18,0.90,22875.0,22,22,13,4,0.590909,14047.88,1650.0,8.513867
9,P002,MetroSpace Tower 02,Abuja,2026-04-30,2026,4,April,20,18,0.90,22875.0,22,22,7,5,0.318182,12719.99,1650.0,7.709085


In [118]:
assert len(query_1_df) == 12
assert len(query_5_df) == 72

print("=" * 60)
print("TOPIC 4 POSTGRESQL MART AND ANALYTICS COMPLETED")
print("=" * 60)

TOPIC 4 POSTGRESQL MART AND ANALYTICS COMPLETED


## 9. Topic 5 — AWS serverless lake and Athena (15 points)

**Create and document the following pipeline in your permitted AWS environment:**

1. Configure a uniquely named **private S3 bucket** with `raw/`, `curated/`, and `audit/` prefixes. Upload a raw CSV to `raw/`; upload **your actual locally generated curated Parquet** to `curated/property_month_kpis/` (preserving partitions). Avoid public ACLs; use encryption defaults and least-privilege IAM.
2. Create a **Python Lambda** triggered by new objects under `raw/`. The handler checks prefix, extension, nonzero size and relevant object metadata, then records an audit JSON under `audit/` and emits structured logs to **CloudWatch**. Add robust handling for event retries and duplicate notifications (e.g., deterministic audit key based on bucket/key/version or eTag and idempotent overwrite), and **prevent a trigger loop** by only triggering on `raw/`.
3. Register the curated Parquet dataset with the **Glue Data Catalog**, through a crawler or explicit schema, and validate inferred partition columns and types. Configure an **Athena query result S3 location** with permissions, and query the catalogued dataset.
4. Run Athena queries for top five property-month occupancy values and portfolio monthly energy totals. Reconcile the monthly energy totals against your local Spark/PostgreSQL results. Capture CloudWatch Lambda logs and an Athena query result as evidence.
5. Explain IAM permissions, S3 event notification behavior, region consistency, expected costs, operational failure modes, and teardown steps. **Do not** give Lambda write permissions across all S3 buckets.

**Division of responsibility:** Lambda validates new raw arrivals and writes an audit; the **Spark computation is performed separately** and its outputs are explicitly uploaded to curated storage. There is no claim that the Lambda event by itself performs the full local Spark pipeline.

**If a course-lab account blocks resource creation:** submit `aws/lambda_handler.py`, architecture diagram, least-privilege IAM pseudopolicy with resource scoping, AWS CLI/console deployment steps, sample S3 event and local Lambda test output, Glue/Athena DDL or crawler configuration, and a clear list of steps you could not execute. Your instructor should apply the same preannounced evidence policy across the cohort.

## Topic 5 Architecture

The AWS serverless lake extends the local PropTech360 data pipeline into cloud storage and serverless analytics.

The curated property-month KPI dataset is generated locally using PySpark and stored as partitioned Parquet files. These files are uploaded to Amazon S3 under the `curated/property_month_kpis/` prefix while retaining their monthly partition structure.

Raw CSV arrivals are uploaded separately under the `raw/` prefix. An Amazon S3 ObjectCreated event triggers the `proptech360-raw-audit` AWS Lambda function only for objects created under this prefix.

The Lambda function validates relevant object metadata and writes an audit JSON document under the `audit/` prefix. Execution logs are automatically written to Amazon CloudWatch Logs.

The curated Parquet dataset is registered in the AWS Glue Data Catalog and queried using Amazon Athena. Athena query results are stored separately in the `athena-results/` prefix.

The Spark transformation is intentionally performed outside Lambda. Lambda is responsible only for validating raw arrivals and generating audit metadata.

## AWS Operational Considerations

### IAM and Least Privilege

The Lambda execution role uses least-privilege access. The function requires permission to inspect objects uploaded under the `raw/` prefix and permission to write audit JSON records only under the `audit/` prefix of the PropTech360 S3 bucket.

The function does not require write access to unrelated S3 buckets or to the curated dataset.

CloudWatch logging permissions are provided through the Lambda basic execution role.

### S3 Event Behaviour

The Lambda function is triggered by S3 ObjectCreated events restricted to the `raw/` prefix.

Restricting the trigger to `raw/` prevents an event loop because audit records written to the `audit/` prefix do not trigger the same Lambda function.

S3 event notifications should be treated as at-least-once delivery. The audit implementation therefore uses a deterministic audit key so that repeated processing of the same logical object replaces the audit record instead of continuously generating duplicate audit files.

### Region Consistency

The S3 bucket, Lambda function, Glue Data Catalog and Athena resources are configured in the same AWS Region, `us-east-1`.

Keeping these resources in the same region simplifies permissions, event integration and data access.

### Expected Costs

The solution uses serverless AWS services with usage-based pricing.

Amazon S3 charges primarily for storage and requests. AWS Lambda charges according to invocations and execution duration. Amazon Athena charges according to the amount of data scanned by queries. AWS Glue Data Catalog costs are minimal for a project of this size.

Because the project dataset contains only 72 curated property-month records, expected development costs are very small.

### Operational Failure Modes

Potential failures include:

- malformed or empty uploaded CSV files;
- objects uploaded outside the expected `raw/` prefix;
- incorrect S3 or Lambda IAM permissions;
- Lambda execution failures;
- missing or incorrect Glue partition metadata;
- incorrect Athena table schema;
- cross-region resource configuration;
- duplicated S3 event notifications.

CloudWatch Logs provide the primary observability mechanism for Lambda execution and troubleshooting.

### Teardown

After the project is complete, cloud resources can be removed to avoid unnecessary charges.

The recommended teardown order is:

1. Remove the S3 event notification.
2. Delete the Lambda function.
3. Delete the Lambda IAM policies or role if no longer required.
4. Delete the Glue table and database if dedicated to this project.
5. Delete Athena query result files.
6. Delete the raw, audit and curated S3 objects.
7. Delete the S3 bucket.

In [ ]:
# STUDENT TASK 5: Minimal AWS Lambda handler outline — save the completed version to aws/lambda_handler.py.
# This handler is NOT run automatically by the notebook. Implement, deploy and test independently.
LAMBDA_HANDLER_STARTER = r''''''
import json
import os
import boto3
from urllib.parse import unquote_plus

s3 = boto3.client('s3')
AUDIT_BUCKET = os.environ.get('AUDIT_BUCKET')

def lambda_handler(event, context):
    # TODO: iterate Records safely; validate s3:ObjectCreated event, raw/ prefix and .csv suffix.
    # TODO: decode the object key, inspect metadata with head_object and reject zero-size files.
    # TODO: derive idempotent audit key from bucket/key/object version or eTag.
    # TODO: write JSON audit under audit/ with scoped S3 permissions; structured logs.
    # TODO: handle or report partial failure without swallowing exceptions.
    raise NotImplementedError('Student implementation required')
''''''
print('Lambda starter ready; copy into aws/lambda_handler.py and complete the TODOs.')

## 10. Testing, deliverables, and presentation (5 points)

**Automated validation checklist:** prove the generator can be rerun; raw hashes stay unchanged; key uniqueness and foreign keys hold after cleaning; quarantine reasons reconcile; Spark emits 72 unique property-month rows; occupancy and SLA rates are between 0 and 1 or NULL when denominator is zero; monthly rent is not multiplied by work-order counts; PostgreSQL repeated upsert retains 72 fact rows; Athena energy totals reconcile with local output for a selected month.

**Submit:** completed `.ipynb`; runnable repository with `README.md`, pinned `requirements.txt`, `src/`, `sql/`, `aws/`, `tests/`, generated synthetic raw data, sample quarantine/curated outputs (or reproducible generation instructions), a short architecture diagram, and redacted evidence. In README, document commands, assumptions, quality decisions, AWS resource names **without credentials**, rerun instructions, and cleanup. Provide a **5–8 minute walkthrough**: the business problem, one intentional data defect, local KPI reconciliation, AWS evidence, and one improvement you would make for production.

**Suggested assessment window:** 10–14 days, individual work or clearly attributed group contributions. Code can be modularized outside the notebook; the notebook should remain the readable end-to-end assessment entry point.

## 11. Grading rubric (100 points)

| Topic | Points | Assessor looks for |
|---|---:|---|
| Python ingestion | 15 | Reusable loader (5); schema/error validation (4); audit digest/metadata (4); rerun safety (2) |
| Pandas data quality | 20 | Profiling and rule coverage (8); explicit quarantine + reasons (6); reconciliation (3); tests (3) |
| PySpark ETL | 25 | Correct grain and snapshot logic (8); SLA and energy transformations (7); robust joins, edge cases and assertions (6); partitioned curated output (4) |
| PostgreSQL and SQL | 20 | Schema and constraints (5); idempotent load (5); five accurate business queries (8); Spark/SQL reconciliation (2) |
| AWS serverless | 15 | Scoped S3 and Lambda event/audit (6); Glue/Athena results (5); IAM/CloudWatch, cost, failure handling and teardown (4) |
| Testing, documentation and demo | 5 | Repeatable tests (2); README/architecture (2); clear demo (1) |
| **Total** | **100** | |

**Assessment principles:** Award points for correct, reproducible and evidenced work, not merely screenshots or copied service configurations. A correct alternative design is acceptable when it meets the stated grain, definitions, validation, and security requirements. Record your institution's policy for restricted AWS lab access before grading.

### Final student reflection

Write 200–300 words: Which source defect most threatened KPI correctness? Why is the property-month fact grain useful? What is one drawback of the serverless design? What would you improve for production scale, monitoring, or data governance?